In [1]:
import datetime as dt
import dask
import numpy as np
import pandas as pd
from typing import Union
from numpy.typing import NDArray
from numba import jit
import random
import dask.dataframe as dd
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster, Semaphore
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Literal, Optional
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import mackinnonp, mackinnoncrit
from statsmodels.tsa.tsatools import lagmat
import statsmodels.api as sm
import warnings

from scipy.integrate import quad
from scipy.special import pbdv, gammaln
from scipy.stats import norm


load_dotenv()

# Infrastructure parameters
POSTGRES_URL = os.getenv("POSTGRES_URL")
CLUSTER_TYPE = "local"
N_WORKERS = 6
N_CONCCURENT_DATABASE_CALLS = 10

# Model parameters
DISCOUNT_RATE = 0.001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
PVALUE_THRESHOLD = 0.001  # Only trade if we have 99.9% confidence
PERCENT_LOSS = 0.10
CASH_ALLOCATION = 100

dask.config.set({"distributed.scheduler.locks.lease-timeout": "120s"})  # 2 minutes

engine = create_engine(POSTGRES_URL)

In [2]:
# Constants: These are the parameters that are used to estimate the GBM and OU processes they should never be changed.
DELTA_T = 1

# Default step size for finite difference approximations in prime calculations
H = 1e-6

# Threshold below which the OU process is considered degenerate.
# When mu < threshold, the process degenerates to Brownian motion (no mean reversion),
# and the F/G integrals diverge due to r/mu terms.
# When sigma < threshold, the process becomes deterministic (no stochastic component).
DEGENERATE_OU_THRESHOLD = 1e-6


class StochasticModelResult(ABC):
    """
    Base class for stochastic model results.
    """

    @abstractmethod
    def __init__(self, params: dict):
        self.params = params

    @abstractmethod
    def to_dict(self) -> dict:
        """
        Convert the results to a dictionary.
        """
        pass


@dataclass
class GeometricBrownianMotionResult(StochasticModelResult):
    """
    Results for the Geometric Brownian Motion (GBM) process.

    Parameters
    ----------
    mu : float
        Drift parameter (μ) in the SDE: dS = mu * S * dt + sigma * S * dW
    sigma : float
        Volatility parameter (σ) in the SDE: dS = mu * S * dt + sigma * S * dW
    """

    mu: float  # drift parameter
    sigma: float  # volatility parameter

    def __init__(self, mu: float, sigma: float):
        self.mu = mu
        self.sigma = sigma
        self.params = {"mu": self.mu, "sigma": self.sigma}

    def to_dict(self) -> dict:
        """
        Convert the results to a dictionary.
        """
        return {"mu": self.mu, "sigma": self.sigma}


@dataclass
class OrnsteinUhlenbeckResult(StochasticModelResult):
    """
    Results for the Ornstein-Uhlenbeck process.

    Parameters
    ----------
    mu : float
        Mean reversion parameter (μ) in the SDE: dX = mu * (theta - X) dt + sigma * dW
    theta : float
        Asymptotic mean (θ) in the SDE: dX = mu * (theta - X) dt + sigma * dW
    sigma : float
        Brownian motion scale (σ) in the SDE: dX = mu * (theta - X) dt + sigma * dW
    """

    mu: float  # mean reversion parameter
    theta: float  # asymptotic mean
    sigma: float  # Brownian motion scale (standard deviation)

    def __init__(
        self,
        mu: float,
        theta: float,
        sigma: float,
    ):
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.params = {
            "mu": self.mu,
            "theta": self.theta,
            "sigma": self.sigma,
        }

    def to_dict(self) -> dict:
        return {
            "mu": self.mu,
            "theta": self.theta,
            "sigma": self.sigma,
        }


class StochasticModel(ABC):
    """
    Base class for stochastic models.
    """

    __params: StochasticModelResult

    def __init__(self, params: StochasticModelResult = None):
        self.__params = params

    @property
    def params(self) -> StochasticModelResult:
        """
        Get the parameters of the model.
        """
        if self.__params is None:
            raise ValueError("Parameters are not set for the model.")
        return self.__params

    @params.setter
    def params(self, params: StochasticModelResult):
        """
        Set the parameters of the model.
        """
        self.__params = params

    @abstractmethod
    def log_likelihood(self, X: np.ndarray) -> float:
        """
        Compute the log likelihood of the model.
        """
        pass

    @abstractmethod
    def fit(self, X: np.ndarray):
        """
        Fit the model to the data.
        """
        pass

    @abstractmethod
    def simulate(self, N: int, N_simulated: int, X_0: float) -> np.ndarray:
        """
        Simulate the model.
        """
        pass


class OrnsteinUhlenbeck(StochasticModel):
    """
    Ornstein-Uhlenbeck process.

    The Ornstein-Uhlenbeck process is defined by:

    dX_t = mu * (theta - X_t) dt + sigma dW_t

    where:
    - mu is the mean reversion rate (speed of mean reversion)
    - theta is the long-term mean (asymptotic mean)
    - sigma is the volatility parameter
    - W_t is a Wiener process

    This matches the standard parameterization used in Leung & Li (2015).
    """

    def __init__(self, params: OrnsteinUhlenbeckResult = None):
        super().__init__(params)

    def log_likelihood(self, X: np.ndarray) -> float:
        """
        Computes the log likelihood of the OU process.

        Uses the global DELTA_T constant for time step.
        """
        # Get the number of observations.
        n = len(X)

        # Get the lag and next values.
        X_lag = X[:-1]
        X_next = X[1:]

        # Get the tilde sigma.
        tilde_sigma = self.params.sigma * np.sqrt(
            (1 - np.exp(-2 * self.params.mu * DELTA_T)) / (2 * self.params.mu)
        )

        # Compute the log likelihood.
        log_likelihood = (
            -0.5 * np.log(2 * np.pi)
            - np.log(tilde_sigma)
            - 1
            / (2 * n * tilde_sigma**2)
            * np.sum(
                (
                    X_next
                    - X_lag * np.exp(-self.params.mu * DELTA_T)
                    - self.params.theta * (1 - np.exp(-self.params.mu * DELTA_T))
                )
                ** 2
            )
        )

        return -log_likelihood

    def fit(self, X: np.ndarray) -> OrnsteinUhlenbeckResult:
        """
        Estimates Ornstein-Uhlenbeck parameters from the given array using OLS regression
        on the exact discrete-time solution (not the Euler approximation).

        The exact OU discrete transition is:
        X_{t+dt} = theta*(1 - exp(-mu*dt)) + X_t*exp(-mu*dt) + sigma*sqrt((1-exp(-2*mu*dt))/(2*mu))*noise

        Letting a = exp(-mu*dt), we can write:
        X_{t+dt} = theta*(1 - a) + X_t*a + noise

        OLS regression of X_{t+1} on X_t gives:
        - intercept = theta*(1 - a)
        - coef = a = exp(-mu*dt)

        Therefore:
        - mu = -log(coef) / dt
        - theta = intercept / (1 - coef)

        input: X - array-like data to be fit as an OU process
        returns: OrnsteinUhlenbeckResult
        """
        # Regress X_{t+1} on X_t (not differences!)
        X_next = X[1:]
        X_lag = X[:-1]
        X_with_const = sm.add_constant(X_lag)

        # Fit OLS regression: X_{t+1} = intercept + coef*X_t
        model = sm.OLS(X_next, X_with_const)
        results = model.fit()

        # Extract coefficients: [intercept, coef]
        intercept = results.params[0]
        coef = results.params[1]

        # Extract OU parameters from exact solution
        # coef = exp(-mu*DELTA_T), which must be in (0, 1) for a valid mean-reverting OU process
        if not (0 < coef < 1):
            raise ValueError(
                f"Invalid OLS coefficient {coef:.6f}. For a mean-reverting OU process, "
                f"the coefficient must be in (0, 1) since it equals exp(-mu*DELTA_T). "
                f"This indicates the data does not follow an OU process or has insufficient variation."
            )

        mu = -np.log(coef) / DELTA_T
        theta = intercept / (1 - coef)

        # Get residual standard deviation
        # residuals = X_{t+1} - (theta*(1-a) + X_t*a)
        # Theoretical: sigma_residual = sigma * sqrt((1 - exp(-2*mu*dt)) / (2*mu))
        residual_std = np.sqrt(results.mse_resid)

        # Back out sigma from residual_std
        # residual_std^2 = sigma^2 * (1 - exp(-2*mu*dt)) / (2*mu)
        sigma = residual_std * np.sqrt(2 * mu / (1 - np.exp(-2 * mu * DELTA_T)))

        # Update the parameters.
        self.params = OrnsteinUhlenbeckResult(mu=mu, theta=theta, sigma=sigma)

        return self.params

    def simulate(self, N: int, N_simulated: int, X_0: float) -> np.ndarray:
        """
        Simulates the OU process.

        Uses the global DELTA_T constant for time step.
        """
        # Initialize the simulated process.
        X_simulated = np.zeros((N_simulated, N))
        X_simulated[:, 0] = X_0  # initial value

        # Simulate the process.
        for i in range(1, N):
            X_simulated[:, i] = (
                X_simulated[:, i - 1] * np.exp(-self.params.mu * DELTA_T)
                + self.params.theta * (1 - np.exp(-self.params.mu * DELTA_T))
                + self.params.sigma
                * np.sqrt(
                    (1 - np.exp(-2 * self.params.mu * DELTA_T)) / (2 * self.params.mu)
                )
                * np.random.normal(0, 1, N_simulated)
            )

        return X_simulated

    @staticmethod
    def f(
        u: float, x: float, mu: float, sigma: float, theta: float, r: float = 0.01
    ) -> float:
        """
        Computes the f function.
        f(u, r) = u^((r/mu) - 1) * e^(sqrt((2*mu)/(sigma^2)) * (x - theta) * u  - u^2 / 2)
        """
        return u ** ((r / mu) - 1) * np.exp(
            np.sqrt((2 * mu) / (sigma**2)) * (x - theta) * u - u**2 / 2
        )

    @staticmethod
    def f_integrand(
        u: float,
        x: float,
        mu: float,
        sigma: float,
        theta: float,
        derivative: int = 0,
        r: float = 0.01,
    ) -> float:
        """
        Computes the f function for various derivatives with respect to x based on the base function f(u, r).
        f(u, r) = u^((r/mu) - 1) * e^(sqrt((2*mu)/(sigma^2)) * (x - theta) * u  - u^2 / 2)
        """
        alpha = (r / mu) - 1 + derivative
        beta = np.sqrt((2 * mu) / (sigma**2)) * (x - theta)
        return (beta**derivative) * (u**alpha) * np.exp(beta * u - u**2 / 2)

    @staticmethod
    def g_integrand(
        u: float,
        x: float,
        mu: float,
        sigma: float,
        theta: float,
        derivative: int = 0,
        r: float = 0.01,
    ) -> float:
        """
        Computes the f function for various derivatives with respect to x based on the base function f(u, r).
        f(u, r) = u^((r/mu) - 1) * e^(sqrt((2*mu)/(sigma^2)) * (x - theta) * u  - u^2 / 2)
        """
        alpha = (r / mu) - 1 + derivative
        beta = np.sqrt((2 * mu) / (sigma**2)) * (theta - x)
        return (beta**derivative) * (u**alpha) * np.exp(beta * u - u**2 / 2)

    @staticmethod
    def f_prime(
        u: float, x: float, mu: float, sigma: float, theta: float, r: float = 0.01
    ) -> float:
        """
        Computes the derivative of f with respect to x.
        f'(u, r) = sqrt((2*mu)/(sigma^2)) * u^((r/mu) - 1) * e^(sqrt((2*mu)/(sigma^2)) * (x - theta) * u  - u^2 / 2)
        """
        return (
            np.sqrt((2 * mu) / (sigma**2))
            * u ** (r / mu)
            * np.exp(np.sqrt((2 * mu) / (sigma**2)) * (x - theta) * u - u**2 / 2)
        )

    @staticmethod
    def F(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float = 0.01,
        derivative: int = 0,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        Computes the F function or its nth derivative with respect to x.
            derivative=0: F(x, r)   = integral f(u, x, r) du from 0 to infinity
            derivative=1: F'(x, r)  = dF/dx
            derivative=2: F''(x, r) = d²F/dx²

        By default uses numerical integration with quad for accuracy.
        An experimental analytical approximation using parabolic cylinder functions
        is available with use_analytical=True, but has ~1-2% error that varies with parameters.

        Parameters
        ----------
        x : float or ndarray
            Absolute position values.
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        theta : float or ndarray
            Mean reversion level(s)
        r : float, optional
            Discount rate (default: 0.01)
        derivative : int, optional
            Order of derivative with respect to x (default: 0)
        use_analytical : bool, optional
            If True, use experimental analytical approximation (faster but less accurate).
            If False, use numerical integration with quad (default, most accurate).

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays

        Notes
        -----
        The integral has a singularity at u=0 when r/mu < 1. The quad integrator
        handles this well using the Wynn epsilon algorithm.

        The analytical approximation provides 10-100x speedup but has accuracy issues
        (~1-2% error) that vary with the parameters, particularly with r/mu ratio.
        Use only if speed is critical and small errors are acceptable.
        """
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        x_arr = np.atleast_1d(x)
        theta_arr = np.atleast_1d(theta)
        mu_arr, sigma_arr, x_arr, theta_arr = np.broadcast_arrays(
            mu_arr, sigma_arr, x_arr, theta_arr
        )

        alpha = (r / mu_arr) - 1
        scale = np.sqrt(2 * mu_arr / sigma_arr**2)
        beta = scale * (x_arr - theta_arr)

        return OrnsteinUhlenbeck._compute_integral(
            alpha, scale, beta, derivative=derivative, use_analytical=use_analytical
        )

    @staticmethod
    def g(
        u: float, x: float, mu: float, sigma: float, theta: float, r: float = 0.01
    ) -> float:
        """
        Computes the g function.
        g(u, x, r) = u^((r/mu) - 1) * e^(sqrt((2*mu)/(sigma^2)) * (theta - x) * u  - u^2 / 2)
        """
        return u ** ((r / mu) - 1) * np.exp(
            np.sqrt((2 * mu) / (sigma**2)) * (theta - x) * u - u**2 / 2
        )

    @staticmethod
    def g_prime(
        u: float, x: float, mu: float, sigma: float, theta: float, r: float = 0.01
    ) -> float:
        """
        Computes the derivative of g with respect to x.
        g'(u, x, r) = -sqrt((2*mu)/(sigma^2)) * u^((r/mu) - 2) * e^(sqrt((2*mu)/(sigma^2)) * (theta - x) * u  - u^2 / 2)
        """
        return (
            -np.sqrt((2 * mu) / (sigma**2))
            * u ** ((r / mu) - 2)
            * np.exp(np.sqrt((2 * mu) / (sigma**2)) * (theta - x) * u - u**2 / 2)
        )

    @staticmethod
    def _compute_integral(
        alpha: float | np.ndarray,
        scale: float | np.ndarray,
        beta: float | np.ndarray,
        derivative: int = 0,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        General-purpose integral of the form:
            scale^derivative * integral_0^inf u^(alpha + derivative) * exp(beta*u - u^2/2) du

        This is the shared computation underlying F, G, and their derivatives.
        The derivative parameter controls both the alpha offset and scaling factor,
        matching the pattern from f_integrand/g_integrand:
            derivative=0: integral u^alpha * exp(beta*u - u^2/2) du           (F, G)
            derivative=1: scale * integral u^(alpha+1) * exp(beta*u - u^2/2) du (F', G')
            derivative=2: scale^2 * integral u^(alpha+2) * exp(beta*u - u^2/2) du (F'', G'')

        Callers provide the base alpha = (r/mu) - 1 and the appropriate scale and beta:
            F family:  scale = sqrt(2mu/sigma^2), beta = scale * (x - theta)
            G family:  scale = sqrt(2mu/sigma^2), beta = scale * (theta - x)

        Parameters
        ----------
        alpha : float or ndarray
            Base power of u in the integrand (before derivative offset)
        scale : float or ndarray
            Scale factor for derivatives (sqrt(2*mu/sigma^2))
        beta : float or ndarray
            Coefficient of u in the exponential
        derivative : int, optional
            Order of derivative (default: 0). Adds to alpha and scales by scale^derivative.
        use_analytical : bool, optional
            If True, use parabolic cylinder function approximation.
            If False, use numerical integration with quad (default).

        Returns
        -------
        float or ndarray
        """
        # Apply derivative: shift alpha and compute scale factor
        effective_alpha = alpha + derivative if derivative != 0 else alpha
        scale_factor = scale**derivative if derivative != 0 else 1.0

        if not use_analytical:
            all_scalar = (
                np.isscalar(effective_alpha)
                and np.isscalar(beta)
                and np.isscalar(scale_factor)
            )

            if all_scalar:
                val = quad(
                    lambda u: u**effective_alpha * np.exp(beta * u - u**2 / 2),
                    0,
                    np.inf,
                )[0]
                return scale_factor * val

            alpha_arr = np.atleast_1d(effective_alpha)
            beta_arr = np.atleast_1d(beta)
            scale_factor_arr = np.atleast_1d(scale_factor)
            alpha_arr, beta_arr, scale_factor_arr = np.broadcast_arrays(
                alpha_arr, beta_arr, scale_factor_arr
            )

            result = np.empty(alpha_arr.shape)
            for idx in np.ndindex(alpha_arr.shape):
                result[idx] = (
                    scale_factor_arr[idx]
                    * quad(
                        lambda u, a=alpha_arr[idx], b=beta_arr[idx]: u**a
                        * np.exp(b * u - u**2 / 2),
                        0,
                        np.inf,
                    )[0]
                )

            if result.ndim == 0:
                return float(result)
            return result

        # Analytical path using parabolic cylinder functions
        # Based on: integral_0^inf u^alpha exp(beta*u - u^2/2) du
        #         = exp(beta^2/4) * Gamma(alpha+1) * D_{-alpha-1}(-beta)
        alpha_arr = np.atleast_1d(effective_alpha)
        beta_arr = np.atleast_1d(beta)
        scale_factor_arr = np.atleast_1d(scale_factor)
        alpha_arr, beta_arr, scale_factor_arr = np.broadcast_arrays(
            alpha_arr, beta_arr, scale_factor_arr
        )

        result = np.empty(alpha_arr.shape, dtype=float)

        alpha_zero_mask = np.isclose(alpha_arr, 0, atol=1e-10)
        non_zero_mask = ~alpha_zero_mask

        # Special case: alpha ~ 0 (exact formula)
        if np.any(alpha_zero_mask):
            result[alpha_zero_mask] = (
                scale_factor_arr[alpha_zero_mask]
                * np.exp(beta_arr[alpha_zero_mask] ** 2 / 2)
                * np.sqrt(2 * np.pi)
                * norm.cdf(beta_arr[alpha_zero_mask])
            )

        if np.any(non_zero_mask):
            a_nz = alpha_arr[non_zero_mask]
            b_nz = beta_arr[non_zero_mask]
            s_nz = scale_factor_arr[non_zero_mask]
            nu_nz = -(a_nz + 1)

            D_vals, _ = pbdv(nu_nz, -b_nz)
            log_prefactor = gammaln(a_nz + 1) + b_nz**2 / 4

            non_zero_result = np.zeros_like(D_vals)
            D_is_nan = np.isnan(D_vals)
            D_is_posinf = np.isposinf(D_vals)
            D_is_neginf = np.isneginf(D_vals)
            D_is_finite = ~(D_is_nan | D_is_posinf | D_is_neginf)

            if np.any(D_is_nan):
                prefactor_is_zero = log_prefactor[D_is_nan] == -np.inf
                non_zero_result[D_is_nan] = np.where(prefactor_is_zero, 0.0, np.nan)

            if np.any(D_is_posinf):
                prefactor_is_zero = log_prefactor[D_is_posinf] == -np.inf
                non_zero_result[D_is_posinf] = np.where(
                    prefactor_is_zero, np.nan, np.inf
                )

            if np.any(D_is_neginf):
                prefactor_is_zero = log_prefactor[D_is_neginf] == -np.inf
                non_zero_result[D_is_neginf] = np.where(
                    prefactor_is_zero, np.nan, -np.inf
                )

            if np.any(D_is_finite):
                prefactor = np.exp(log_prefactor[D_is_finite])
                non_zero_result[D_is_finite] = prefactor * D_vals[D_is_finite]
                finite_overflow = ~np.isfinite(non_zero_result[D_is_finite])
                if np.any(finite_overflow):
                    indices = np.where(D_is_finite)[0][finite_overflow]
                    signs = np.sign(prefactor[finite_overflow]) * np.sign(
                        D_vals[indices]
                    )
                    non_zero_result[indices] = signs * np.inf

            result[non_zero_mask] = s_nz * non_zero_result

        if result.ndim == 0:
            return float(result)
        return result

    @staticmethod
    def G(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float = 0.01,
        derivative: int = 0,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        Computes the G function or its nth derivative with respect to x.
            derivative=0: G(x, r)   = integral g(u, x, r) du from 0 to infinity
            derivative=1: G'(x, r)  = dG/dx
            derivative=2: G''(x, r) = d²G/dx²

        By default uses numerical integration with quad for accuracy.
        An experimental analytical approximation using parabolic cylinder functions
        is available with use_analytical=True, but has ~1-2% error that varies with parameters.

        Parameters
        ----------
        x : float or ndarray
            Absolute position values.
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        theta : float or ndarray
            Mean reversion level(s)
        r : float, optional
            Discount rate (default: 0.01)
        derivative : int, optional
            Order of derivative with respect to x (default: 0)
        use_analytical : bool, optional
            If True, use experimental analytical approximation (faster but less accurate).
            If False, use numerical integration with quad (default, most accurate).

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays

        Notes
        -----
        The integral has a singularity at u=0 when r/mu < 1. The quad integrator
        handles this well using the Wynn epsilon algorithm.

        The analytical approximation provides 10-100x speedup but has accuracy issues
        (~1-2% error) that vary with the parameters. Use only if speed is critical
        and small errors are acceptable.
        """
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        x_arr = np.atleast_1d(x)
        theta_arr = np.atleast_1d(theta)
        mu_arr, sigma_arr, x_arr, theta_arr = np.broadcast_arrays(
            mu_arr, sigma_arr, x_arr, theta_arr
        )

        alpha = (r / mu_arr) - 1
        scale = -1 * np.sqrt(2 * mu_arr / sigma_arr**2)
        beta = scale * (x_arr - theta_arr)

        return OrnsteinUhlenbeck._compute_integral(
            alpha, scale, beta, derivative=derivative, use_analytical=use_analytical
        )

    @staticmethod
    def V(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The value function V(x, r).

        Parameters
        ----------
        x : float or ndarray
            Absolute position values.
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        theta : float or ndarray
            Mean reversion level(s)
        r : float or ndarray
            Discount rate(s)
        c : float or ndarray
            Transaction cost(s)
        exit_level : float or ndarray
            Exit level(s) as absolute positions
        use_analytical : bool, optional
            If True, use analytical F function (default: False)
        """
        # Convert to arrays and broadcast to same shape
        x_arr = np.atleast_1d(x)
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        theta_arr = np.atleast_1d(theta)
        r_arr = np.atleast_1d(r)
        c_arr = np.atleast_1d(c)
        exit_level_arr = np.atleast_1d(exit_level)

        # Broadcast all arrays to the same shape
        x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr = (
            np.broadcast_arrays(
                x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr
            )
        )

        # Create masks for element-wise operations
        left_mask = x_arr < exit_level_arr

        # Compute V_left for all elements (will be used where left_mask is True)
        F_x = OrnsteinUhlenbeck.F(
            x_arr, mu_arr, sigma_arr, theta_arr, r_arr, use_analytical=use_analytical
        )
        F_exit = OrnsteinUhlenbeck.F(
            exit_level_arr,
            mu_arr,
            sigma_arr,
            theta_arr,
            r_arr,
            use_analytical=use_analytical,
        )
        V_left = ((exit_level_arr - c_arr) / F_exit) * F_x

        # Compute V_right for all elements (will be used where left_mask is False)
        V_right = x_arr - c_arr

        # Use np.where for element-wise selection
        result = np.where(left_mask, V_left, V_right)

        # Return scalar if input was scalar
        if (
            np.isscalar(x)
            and np.isscalar(mu)
            and np.isscalar(sigma)
            and np.isscalar(theta)
            and np.isscalar(r)
            and np.isscalar(c)
            and np.isscalar(exit_level)
        ):
            return float(result)

        return result

    @staticmethod
    def V_prime(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The derivative of the value function V(x, r) with respect to x.

        For x < exit_level: V'(x) = [(exit_level - c) / F(exit_level)] * F'(x)
        For x >= exit_level: V'(x) = 1

        Parameters
        ----------
        x : float or ndarray
            Absolute position values.
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        theta : float or ndarray
            Mean reversion level(s)
        r : float or ndarray
            Discount rate(s)
        c : float or ndarray
            Transaction cost(s)
        exit_level : float or ndarray
            Exit level(s) as absolute positions
        use_analytical : bool, optional
            If True, use analytical F function (default: False)

        Returns
        -------
        float or ndarray
            Derivative dV/dx
        """
        # Convert to arrays and broadcast to same shape
        x_arr = np.atleast_1d(x)
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        theta_arr = np.atleast_1d(theta)
        r_arr = np.atleast_1d(r)
        c_arr = np.atleast_1d(c)
        exit_level_arr = np.atleast_1d(exit_level)

        # Broadcast all arrays to the same shape
        x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr = (
            np.broadcast_arrays(
                x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr
            )
        )

        # Create masks for element-wise operations
        left_mask = x_arr < exit_level_arr

        # Compute V'_left for all elements (will be used where left_mask is True)
        # V'(x) = [(exit_level - c) / F(exit_level)] * F'(x)
        F_prime_x = OrnsteinUhlenbeck.F(
            x_arr,
            mu_arr,
            sigma_arr,
            theta_arr,
            r_arr,
            derivative=1,
            use_analytical=use_analytical,
        )
        F_exit = OrnsteinUhlenbeck.F(
            exit_level_arr,
            mu_arr,
            sigma_arr,
            theta_arr,
            r_arr,
            use_analytical=use_analytical,
        )
        V_prime_left = ((exit_level_arr - c_arr) / F_exit) * F_prime_x

        # For x >= exit_level: V'(x) = d/dx(x - c) = 1
        V_prime_right = np.ones_like(x_arr)

        # Use np.where for element-wise selection
        result = np.where(left_mask, V_prime_left, V_prime_right)

        # Return scalar if input was scalar
        if (
            np.isscalar(x)
            and np.isscalar(mu)
            and np.isscalar(sigma)
            and np.isscalar(theta)
            and np.isscalar(r)
            and np.isscalar(c)
            and np.isscalar(exit_level)
        ):
            return float(result)

        return result

    @staticmethod
    def V_double_prime(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The second derivative of the value function V(x, r) with respect to x.

        For x < exit_level: V''(x) = [(exit_level - c) / F(exit_level)] * F''(x)
        For x >= exit_level: V''(x) = 0

        Parameters
        ----------
        x : float or ndarray
            Absolute position values.
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        theta : float or ndarray
            Mean reversion level(s)
        r : float or ndarray
            Discount rate(s)
        c : float or ndarray
            Transaction cost(s)
        exit_level : float or ndarray
            Exit level(s) as absolute positions
        use_analytical : bool, optional
            If True, use analytical F function (default: False)

        Returns
        -------
        float or ndarray
            Second derivative d²V/dx²
        """
        # Convert to arrays and broadcast to same shape
        x_arr = np.atleast_1d(x)
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        theta_arr = np.atleast_1d(theta)
        r_arr = np.atleast_1d(r)
        c_arr = np.atleast_1d(c)
        exit_level_arr = np.atleast_1d(exit_level)

        # Broadcast all arrays to the same shape
        x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr = (
            np.broadcast_arrays(
                x_arr, mu_arr, sigma_arr, theta_arr, r_arr, c_arr, exit_level_arr
            )
        )

        # Create masks for element-wise operations
        left_mask = x_arr < exit_level_arr

        # Compute V''_left for all elements (will be used where left_mask is True)
        # V''(x) = [(exit_level - c) / F(exit_level)] * F''(x)
        F_double_prime_x = OrnsteinUhlenbeck.F(
            x_arr,
            mu_arr,
            sigma_arr,
            theta_arr,
            r_arr,
            derivative=2,
            use_analytical=use_analytical,
        )
        F_exit = OrnsteinUhlenbeck.F(
            exit_level_arr,
            mu_arr,
            sigma_arr,
            theta_arr,
            r_arr,
            use_analytical=use_analytical,
        )
        V_double_prime_left = (exit_level_arr - c_arr) * F_double_prime_x / F_exit

        # For x >= exit_level: V''(x) = d²/dx²(x - c) = 0
        V_double_prime_right = np.zeros_like(x_arr)

        # Use np.where for element-wise selection
        result = np.where(left_mask, V_double_prime_left, V_double_prime_right)

        # Return scalar if input was scalar
        if (
            np.isscalar(x)
            and np.isscalar(mu)
            and np.isscalar(sigma)
            and np.isscalar(theta)
            and np.isscalar(r)
            and np.isscalar(c)
            and np.isscalar(exit_level)
        ):
            return float(result)

        return result

    @staticmethod
    def C(
        c: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        r: float,
        L: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The constant C in the OU process.

        Parameters
        ----------
        c : float or ndarray
            Transaction cost parameter(s)
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        r : float
            Discount rate
        L : float or ndarray
            Loss level as spread (L - theta)
        exit_level : float or ndarray
            Exit level as spread (exit_level - theta)
        use_analytical : bool, optional
            If True, use analytical F and G functions (default: False)

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays
        """
        return (
            (exit_level - c)
            * OrnsteinUhlenbeck.G(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
            - (L - c)
            * OrnsteinUhlenbeck.G(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
        ) / (
            OrnsteinUhlenbeck.F(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
            * OrnsteinUhlenbeck.G(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
            - OrnsteinUhlenbeck.F(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
            * OrnsteinUhlenbeck.G(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
        )

    @staticmethod
    def D(
        c: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        r: float,
        L: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The optimal entry level d_D^*.

        Parameters
        ----------
        c : float or ndarray
            Transaction cost parameter(s)
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        r : float
            Discount rate
        L : float or ndarray
            Loss level as spread (L - theta)
        exit_level : float or ndarray
            Exit level as spread (exit_level - theta)
        use_analytical : bool, optional
            If True, use analytical F and G functions (default: False)

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays
        """
        return (
            (L - c)
            * OrnsteinUhlenbeck.F(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
            - (exit_level - c)
            * OrnsteinUhlenbeck.F(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
        ) / (
            OrnsteinUhlenbeck.F(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
            * OrnsteinUhlenbeck.G(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
            - OrnsteinUhlenbeck.F(
                L, mu=mu, sigma=sigma, theta=0, r=r, use_analytical=use_analytical
            )
            * OrnsteinUhlenbeck.G(
                exit_level,
                mu=mu,
                sigma=sigma,
                theta=0,
                r=r,
                use_analytical=use_analytical,
            )
        )

    @staticmethod
    def V_L(
        x: float | np.ndarray,
        c: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        r: float,
        L: float | np.ndarray,
        exit_level: float | np.ndarray,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The value function V_L(x, r).

        Parameters
        ----------
        x : float or ndarray
            Spread values (x - theta). MUST be pre-normalized relative to theta.
        c : float or ndarray
            Transaction cost parameter(s)
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        r : float
            Discount rate
        L : float or ndarray
            Loss level as spread (L - theta)
        exit_level : float or ndarray
            Exit level as spread (exit_level - theta)
        use_analytical : bool, optional
            If True, use analytical F and G functions (default: False)

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays
        """
        # Convert to arrays for element-wise operations
        x_arr = np.atleast_1d(x)
        c_arr = np.atleast_1d(c)
        mu_arr = np.atleast_1d(mu)
        sigma_arr = np.atleast_1d(sigma)
        L_arr = np.atleast_1d(L)
        exit_level_arr = np.atleast_1d(exit_level)

        # Broadcast all arrays to the same shape
        x_arr, c_arr, mu_arr, sigma_arr, L_arr, exit_level_arr = np.broadcast_arrays(
            x_arr, c_arr, mu_arr, sigma_arr, L_arr, exit_level_arr
        )

        # Element-wise condition: x < exit_level and x > L
        condition = (x_arr < exit_level_arr) & (x_arr > L_arr)

        # Compute both branches
        waiting_value = OrnsteinUhlenbeck.C(
            c=c_arr,
            mu=mu_arr,
            sigma=sigma_arr,
            r=r,
            L=L_arr,
            exit_level=exit_level_arr,
            use_analytical=use_analytical,
        ) * OrnsteinUhlenbeck.F(
            x_arr,
            mu=mu_arr,
            sigma=sigma_arr,
            theta=0,
            r=r,
            use_analytical=use_analytical,
        ) + OrnsteinUhlenbeck.D(
            c=c_arr,
            mu=mu_arr,
            sigma=sigma_arr,
            r=r,
            L=L_arr,
            exit_level=exit_level_arr,
            use_analytical=use_analytical,
        ) * OrnsteinUhlenbeck.G(
            x_arr,
            mu=mu_arr,
            sigma=sigma_arr,
            theta=0,
            r=r,
            use_analytical=use_analytical,
        )
        immediate_value = x_arr - c_arr

        # Use np.where for element-wise selection
        result = np.where(condition, waiting_value, immediate_value)

        # Return scalar if input was scalar
        if (
            np.isscalar(x)
            and np.isscalar(c)
            and np.isscalar(mu)
            and np.isscalar(sigma)
            and np.isscalar(L)
            and np.isscalar(exit_level)
        ):
            return float(result)

        return result

    @staticmethod
    def V_L_prime(
        x: float | np.ndarray,
        c: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        r: float,
        L: float | np.ndarray,
        exit_level: float | np.ndarray,
        h: float | None = None,
        use_analytical: bool = True,
    ) -> float | np.ndarray:
        """
        The derivative of the value function V_L(x, r).

        Uses finite difference approximation: (V_L(x+h) - V_L(x)) / h

        Parameters
        ----------
        x : float or ndarray
            Spread values (x - theta). MUST be pre-normalized relative to theta.
        c : float or ndarray
            Transaction cost parameter(s)
        mu : float or ndarray
            Mean reversion speed parameter(s)
        sigma : float or ndarray
            Volatility parameter(s)
        r : float
            Discount rate
        L : float or ndarray
            Loss level as spread (L - theta)
        exit_level : float or ndarray
            Exit level as spread (exit_level - theta)
        h : float or None, optional
            Step size for finite difference (default: None, uses module constant H)
        use_analytical : bool, optional
            If True, use analytical F and G functions (default: False)

        Returns
        -------
        float or ndarray
            Scalar if all inputs are scalar, otherwise array with shape
            determined by broadcasting the input arrays
        """
        if h is None:
            h = H
        return (
            OrnsteinUhlenbeck.V_L(
                x + h,
                c=c,
                mu=mu,
                sigma=sigma,
                r=r,
                L=L,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
            - OrnsteinUhlenbeck.V_L(
                x,
                c=c,
                mu=mu,
                sigma=sigma,
                r=r,
                L=L,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
        ) / h

    @staticmethod
    def get_optimal_exit_level(
        mu: np.ndarray,
        sigma: np.ndarray,
        theta: np.ndarray,
        discount_rate: float = 0.01,
        transaction_cost: float = 0.01,
        max_iter: int = 1000,
        tol: float = 1e-6,
        n_grid: int = 1000,
        degenerate_threshold: float = DEGENERATE_OU_THRESHOLD,
    ):
        """
        Vectorized computation of the Ornstein-Uhlenbeck optimal exit level given process parameters.

        Uses a grid search over spreads to find initial guesses, then refines with Newton's method.

        All arguments must be 1D numpy arrays of the same shape.
        Returns: 1D numpy array of exit_levels, or np.nan for positions where
        mu or sigma < degenerate_threshold (degenerate OU process) or any input is NaN/inf.
        """
        mu = np.asarray(mu)
        sigma = np.asarray(sigma)
        theta = np.asarray(theta)

        # Initialize result with NaN
        result = np.full_like(mu, np.nan)

        # Filter out degenerate OU processes and invalid inputs.
        # mu < threshold: degenerates to Brownian motion (no mean reversion, r/mu diverges)
        # sigma < threshold: degenerates to deterministic process (no stochastic component)
        valid = (
            (mu >= degenerate_threshold)
            & (sigma >= degenerate_threshold)
            & np.isfinite(mu)
            & np.isfinite(sigma)
            & np.isfinite(theta)
        )
        if not np.any(valid):
            return result

        mu_v = mu[valid]
        sigma_v = sigma[valid]
        theta_v = theta[valid]
        r = discount_rate
        c = transaction_cost

        # Function f(x) operating on absolute positions
        def f_exit_level(x, mu, sigma, theta, r, c):
            return (x - c) * OrnsteinUhlenbeck.F(
                x, mu, sigma, theta=theta, r=r, derivative=1, use_analytical=True
            ) - OrnsteinUhlenbeck.F(x, mu, sigma, theta=theta, r=r, use_analytical=True)

        # Derivative of f(x) using analytical second derivative
        def f_prime_exit_level(x, mu, sigma, theta, r, c):
            return (x - c) * OrnsteinUhlenbeck.F(
                x, mu, sigma, theta=theta, r=r, derivative=2, use_analytical=True
            )

        # Grid search for initial guesses (in absolute position space)
        # For single parameter set, create 1D grid; for multiple, create 2D grid
        if len(mu_v) == 1:
            # Single parameter: create 1D grid for efficiency
            x_grid = np.linspace(float(theta_v), float(theta_v + 100 * sigma_v), n_grid)
            f_grid = f_exit_level(x_grid, mu_v, sigma_v, theta_v, r, c)
            fp_grid = f_prime_exit_level(x_grid, mu_v, sigma_v, theta_v, r, c)
            # Filter: f(x) > 0 and f'(x) > 0 (both must be positive)
            f_search = np.where((f_grid > 0) & (fp_grid > 0), f_grid, np.inf)
            x0 = np.array([x_grid[np.argmin(f_search)]])
        else:
            # Multiple parameters: create 2D grid
            # x_grid shape: (n_grid, n_params)
            x_grid = np.linspace(theta_v, theta_v + 100 * sigma_v, n_grid)
            f_grid = f_exit_level(x_grid, mu_v, sigma_v, theta_v, r, c)
            fp_grid = f_prime_exit_level(x_grid, mu_v, sigma_v, theta_v, r, c)
            # Filter: f(x) > 0 and f'(x) > 0 (both must be positive)
            f_search = np.where((f_grid > 0) & (fp_grid > 0), f_grid, np.inf)
            x0 = x_grid[np.argmin(f_search, axis=0), np.arange(len(mu_v))]

        # Newton's method
        x = x0
        active = np.ones(len(x), dtype=bool)
        for _ in range(max_iter):
            if not np.any(active):
                break
            f_val = f_exit_level(
                x[active], mu_v[active], sigma_v[active], theta_v[active], r, c
            )
            fp_val = f_prime_exit_level(
                x[active], mu_v[active], sigma_v[active], theta_v[active], r, c
            )
            safe = (fp_val != 0) & np.isfinite(fp_val) & np.isfinite(f_val)
            step = np.zeros_like(f_val)
            np.divide(f_val, fp_val, out=step, where=safe)
            x[active] = x[active] - step
            active_idx = np.flatnonzero(active)
            # NaN out elements that hit invalid values
            x[active_idx[~safe]] = np.nan
            done = (np.abs(f_val) < tol) | ~safe
            active[active_idx[done]] = False

        # Set non-converged values to NaN
        n_unconverged = np.sum(active)
        if n_unconverged > 0:
            warnings.warn(
                f"get_optimal_exit_level: {n_unconverged} roots did not converge "
                f"within {max_iter} iterations."
            )
            x[active] = np.nan

        # Result is already in absolute positions
        result[valid] = x
        return result

    @staticmethod
    def get_optimal_entry_level(
        mu: np.ndarray,
        sigma: np.ndarray,
        theta: np.ndarray,
        exit_level: np.ndarray,
        discount_rate: float,
        transaction_cost: float,
        max_iter: int = 1000,
        tol: float = 1e-6,
        n_grid: int = 1000,
        degenerate_threshold: float = DEGENERATE_OU_THRESHOLD,
    ):
        """
        Compute optimal entry level using grid search for initial guess then Newton's method.

        All arguments must be 1D numpy arrays of the same shape.
        Returns: 1D numpy array of entry levels (absolute values), or np.nan where
        mu or sigma < degenerate_threshold (degenerate OU process) or any input is NaN/inf.
        """
        mu = np.asarray(mu)
        sigma = np.asarray(sigma)
        theta = np.asarray(theta)
        exit_level = np.asarray(exit_level)

        # Initialize result with NaN
        result = np.full_like(mu, np.nan)

        # Filter out degenerate OU processes and invalid inputs.
        # mu < threshold: degenerates to Brownian motion (no mean reversion, r/mu diverges)
        # sigma < threshold: degenerates to deterministic process (no stochastic component)
        valid = (
            (mu >= degenerate_threshold)
            & (sigma >= degenerate_threshold)
            & np.isfinite(mu)
            & np.isfinite(sigma)
            & np.isfinite(theta)
            & np.isfinite(exit_level)
        )
        if not np.any(valid):
            return result

        mu_v = mu[valid]
        sigma_v = sigma[valid]
        theta_v = theta[valid]
        exit_level_v = exit_level[valid]
        r = discount_rate
        c = transaction_cost

        # Function f(x) operating on absolute positions
        def f_entry_level(x, mu, sigma, theta, r, c, exit_level):
            return (
                OrnsteinUhlenbeck.G(
                    x, mu, sigma, theta=theta, r=r, derivative=1, use_analytical=True
                )
                * (
                    OrnsteinUhlenbeck.V(
                        x,
                        mu,
                        sigma,
                        theta=theta,
                        r=r,
                        c=c,
                        exit_level=exit_level,
                        use_analytical=True,
                    )
                    - x
                    - c
                )
            ) - (
                OrnsteinUhlenbeck.G(x, mu, sigma, theta=theta, r=r, use_analytical=True)
                * (
                    OrnsteinUhlenbeck.V_prime(
                        x,
                        mu,
                        sigma,
                        theta=theta,
                        r=r,
                        c=c,
                        exit_level=exit_level,
                        use_analytical=True,
                    )
                    - 1
                )
            )

        # Derivative of f(x) using analytical second derivative
        def f_prime_entry_level(x, mu, sigma, theta, r, c, exit_level):
            return (
                OrnsteinUhlenbeck.G(
                    x, mu, sigma, theta=theta, r=r, derivative=2, use_analytical=True
                )
                * (
                    OrnsteinUhlenbeck.V(
                        x,
                        mu,
                        sigma,
                        theta=theta,
                        r=r,
                        c=c,
                        exit_level=exit_level,
                        use_analytical=True,
                    )
                    - x
                    - c
                )
            ) - (
                OrnsteinUhlenbeck.G(x, mu, sigma, theta=theta, r=r, use_analytical=True)
                * (
                    OrnsteinUhlenbeck.V_double_prime(
                        x,
                        mu,
                        sigma,
                        theta=theta,
                        r=r,
                        c=c,
                        exit_level=exit_level,
                        use_analytical=True,
                    )
                )
            )

        # Grid search for initial guesses (in absolute position space)
        # For single parameter set, create 1D grid; for multiple, create 2D grid
        if len(mu_v) == 1:
            # Single parameter: create 1D grid for efficiency
            x_grid = np.linspace(float(theta_v - 100 * sigma_v), float(theta_v), n_grid)
            f_grid = f_entry_level(x_grid, mu_v, sigma_v, theta_v, r, c, exit_level_v)
            fp_grid = f_prime_entry_level(
                x_grid, mu_v, sigma_v, theta_v, r, c, exit_level_v
            )
            # Filter: f(x) < 0 and f'(x) > 0
            f_search = np.where((f_grid < 0) & (fp_grid > 0), f_grid, np.inf)
            x0 = np.array([x_grid[np.argmin(f_search)]])
        else:
            # Multiple parameters: create 2D grid
            # x_grid shape: (n_grid, n_params)
            x_grid = np.linspace(theta_v - 100 * sigma_v, theta_v, n_grid)
            f_grid = f_entry_level(x_grid, mu_v, sigma_v, theta_v, r, c, exit_level_v)
            fp_grid = f_prime_entry_level(
                x_grid, mu_v, sigma_v, theta_v, r, c, exit_level_v
            )
            # Filter: f(x) < 0 and f'(x) > 0
            f_search = np.where((f_grid < 0) & (fp_grid > 0), f_grid, np.inf)
            x0 = x_grid[np.argmin(f_search, axis=0), np.arange(len(mu_v))]

        # Newton's method
        x = x0
        active = np.ones(len(x), dtype=bool)
        for _ in range(max_iter):
            if not np.any(active):
                break
            f_val = f_entry_level(
                x[active],
                mu_v[active],
                sigma_v[active],
                theta_v[active],
                r,
                c,
                exit_level_v[active],
            )
            fp_val = f_prime_entry_level(
                x[active],
                mu_v[active],
                sigma_v[active],
                theta_v[active],
                r,
                c,
                exit_level_v[active],
            )
            safe = (fp_val != 0) & np.isfinite(fp_val) & np.isfinite(f_val)
            step = np.zeros_like(f_val)
            np.divide(f_val, fp_val, out=step, where=safe)
            x[active] = x[active] - step
            active_idx = np.flatnonzero(active)
            # NaN out elements that hit invalid values
            x[active_idx[~safe]] = np.nan
            done = (np.abs(f_val) < tol) | ~safe
            active[active_idx[done]] = False

        # Set non-converged values to NaN
        n_unconverged = np.sum(active)
        if n_unconverged > 0:
            warnings.warn(
                f"get_optimal_entry_level: {n_unconverged} roots did not converge "
                f"within {max_iter} iterations."
            )
            x[active] = np.nan

        # Result is already in absolute positions
        result[valid] = x
        return result


@dataclass
class RollingCointegrationResults:
    """
    Results from rolling cointegration test.
    """

    beta: NDArray[np.float64]  # 1D array of float64 - hedge ratio
    coint_t: NDArray[np.float64]  # 1D array of float64
    pvalue: NDArray[np.float64]  # 1D array of float64
    crit_1pct: NDArray[np.float64]  # 1D array of float64
    crit_5pct: NDArray[np.float64]  # 1D array of float64
    crit_10pct: NDArray[np.float64]  # 1D array of float64
    residual_mean: NDArray[
        np.float64
    ]  # 1D array of float64 - mean of residuals (spread) for each window
    residual_std: NDArray[
        np.float64
    ]  # 1D array of float64 - standard deviation of residuals (spread) for each window
    usedlag: int  # Number of lags used in ADF test


@dataclass
class RollingOrnsteinUhlenbeckResults:
    """
    Results from rolling Ornstein-Uhlenbeck parameter estimation.
    """

    mu: NDArray[np.float64]  # 1D array of float64
    theta: NDArray[np.float64]  # 1D array of float64
    sigma: NDArray[np.float64]  # 1D array of float64
    half_life: NDArray[np.float64]  # 1D array of float64


def _build_adf_matrices(
    residuals: NDArray[np.float64], lag: int
) -> tuple[NDArray[np.float64], NDArray[np.float64]]:
    """
    Build the matrices needed for ADF regression on residuals.

    ADF regression (no trend): Δresid = ρ * resid_{t-1} + Σ γ_i * Δresid_{t-i} + error

    Parameters
    ----------
    residuals : ndarray
        Residuals from cointegrating regression
    lag : int
        Number of lagged differences to include

    Returns
    -------
    y : ndarray
        Dependent variable (differenced residuals)
    X : ndarray
        Regressors: [lagged_level, lagged_differences]
    """
    # Compute differences
    diff_resid = np.diff(residuals)

    # Build lag matrix of differences (trim='both' keeps aligned observations)
    # This creates [diff_t, diff_{t-1}, ..., diff_{t-lag}]
    xdall = lagmat(diff_resid[:, None], lag, trim="both", original="in")

    # Number of usable observations
    nobs = xdall.shape[0]

    # Replace first column (current diff) with lagged level
    # We want resid[lag:nobs+lag] as the lagged level
    xdall[:, 0] = residuals[lag : nobs + lag]

    # Dependent variable: current differenced residual
    y = diff_resid[lag:]

    # X matrix: [lagged_level, lagged_diffs]
    # Column 0 is lagged level, columns 1:lag+1 are lagged differences
    X = xdall[:, : lag + 1]

    return y, X


class RollingCointegration:
    """
    Rolling cointegration test with no intercept (hedge ratio only).
    """

    def __init__(
        self,
        y0: NDArray[np.float64],
        y1: NDArray[np.float64],
        window: int,
        trend: Literal["c", "ct", "ctt", "n"] = "c",
        lag: int = 1,
        min_nobs: Optional[int] = None,
        expanding: bool = False,
    ):
        self.y0 = y0
        self.y1 = y1
        self.window = window
        self.trend = trend
        self.lag = lag
        self.min_nobs = min_nobs
        self.expanding = expanding

    def fit(
        self, method: Literal["inv", "lstsq", "pinv"] = "inv"
    ) -> RollingCointegrationResults:
        """
        Fit the rolling cointegration model.

        Performs rolling cointegrating regression (no intercept) and ADF test on residuals.
        Model: y0 = beta * y1 + residuals (spread)
        """
        # Input validation
        y0 = np.asarray(self.y0, dtype=np.float64).squeeze()
        y1 = np.asarray(self.y1, dtype=np.float64).squeeze()

        if y0.ndim != 1 or y1.ndim != 1:
            raise ValueError("y0 and y1 must be 1-dimensional")
        if len(y0) != len(y1):
            raise ValueError("y0 and y1 must have the same length")

        nobs = len(y0)
        k_vars = 2  # Two variables for cointegration

        if self.window < self.lag + 5:
            raise ValueError(f"window must be at least lag + 5 = {self.lag + 5}")

        # =========================================================================
        # STAGE 1: Rolling cointegrating regression (NO INTERCEPT)
        # y0 = beta*y1 + residuals (spread)
        # =========================================================================

        # Prepare exogenous: just [y1] with no constant
        exog = y1.reshape(-1, 1)

        # Run RollingOLS for cointegrating regression (no intercept)
        rolling_coint_model = RollingOLS(
            endog=y0,
            exog=exog,
            window=self.window,
            min_nobs=self.min_nobs,
            expanding=self.expanding,
        )
        rolling_coint_results = rolling_coint_model.fit(
            method=method, params_only=False
        )

        # Extract hedge ratio (beta)
        params = rolling_coint_results.params
        beta = params.squeeze().copy()  # Hedge ratio on y1

        # =========================================================================
        # STAGE 2: Rolling ADF test on spread (cointegration test)
        # =========================================================================

        # Initialize output arrays
        coint_t = np.full(nobs, np.nan)
        pvalue = np.full(nobs, np.nan)
        crit_1pct = np.full(nobs, np.nan)
        crit_5pct = np.full(nobs, np.nan)
        crit_10pct = np.full(nobs, np.nan)
        residual_mean = np.full(nobs, np.nan)
        residual_std = np.full(nobs, np.nan)

        # Determine starting index
        if self.expanding:
            first_idx = max(
                self.min_nobs if self.min_nobs is not None else 2, self.lag + 5
            )
        else:
            first_idx = self.window

        # Precompute critical values for cointegration test
        if self.trend != "n":
            adf_nobs_approx = self.window - self.lag - 1
            crit_vals = mackinnoncrit(
                N=k_vars, regression=self.trend, nobs=adf_nobs_approx - 1
            )
            crit_1pct_val = crit_vals[0]
            crit_5pct_val = crit_vals[1]
            crit_10pct_val = crit_vals[2]
        else:
            crit_1pct_val = crit_5pct_val = crit_10pct_val = np.nan

        # Main loop: For each window, compute spread and ADF test
        for t in range(first_idx - 1, nobs):
            # Skip if parameters not available
            if np.isnan(beta[t]):
                continue

            # Get window bounds
            if self.expanding:
                w_start = 0
            else:
                w_start = t - self.window + 1
            w_end = t + 1

            # Compute spread for this window using THIS window's beta (no alpha)
            y0_window = y0[w_start:w_end]
            y1_window = y1[w_start:w_end]
            spread_window = y0_window - beta[t] * y1_window

            # Compute residual statistics
            residual_mean[t] = np.mean(spread_window)
            residual_std[t] = np.std(spread_window, ddof=1)

            # Check for degenerate cases
            if residual_std[t] < 1e-10:
                continue

            # -----------------------------------------------------------------
            # ADF Test on spread (Cointegration Test)
            # -----------------------------------------------------------------
            try:
                adf_y, adf_X = _build_adf_matrices(spread_window, self.lag)
                adf_nobs = len(adf_y)

                if adf_nobs >= self.lag + 2:
                    # Run ADF OLS
                    XtX = adf_X.T @ adf_X
                    Xty = adf_X.T @ adf_y
                    XtX_inv = np.linalg.inv(XtX)
                    adf_params = XtX_inv @ Xty

                    # Compute t-statistic
                    adf_resid = adf_y - adf_X @ adf_params
                    ssr = np.sum(adf_resid**2)
                    df = adf_nobs - adf_X.shape[1]
                    mse = ssr / df
                    se = np.sqrt(np.diag(XtX_inv) * mse)
                    t_stat = adf_params[0] / se[0]

                    coint_t[t] = t_stat
                    pvalue[t] = mackinnonp(t_stat, regression=self.trend, N=k_vars)

                    # Store critical values
                    if self.trend != "n":
                        crit_1pct[t] = crit_1pct_val
                        crit_5pct[t] = crit_5pct_val
                        crit_10pct[t] = crit_10pct_val

            except (np.linalg.LinAlgError, Exception):
                pass

        return RollingCointegrationResults(
            beta=beta,
            coint_t=coint_t,
            pvalue=pvalue,
            crit_1pct=crit_1pct,
            crit_5pct=crit_5pct,
            crit_10pct=crit_10pct,
            residual_mean=residual_mean,
            residual_std=residual_std,
            usedlag=self.lag,
        )


class RollingOrnsteinUhlenbeck(OrnsteinUhlenbeck):
    """
    Rolling Ornstein-Uhlenbeck parameter estimation (no intercept).
    """

    def __init__(
        self,
        beta: NDArray[np.float64],
        y0: NDArray[np.float64],
        y1: NDArray[np.float64],
        window: int,
        lag: int = 1,
        min_nobs: Optional[int] = None,
        expanding: bool = False,
    ):
        # Don't call super().__init__() since we're not using params-based initialization
        self.beta = beta
        self.y0 = y0
        self.y1 = y1
        self.window = window
        self.lag = lag
        self.min_nobs = min_nobs
        self.expanding = expanding

    def fit(self) -> RollingOrnsteinUhlenbeckResults:
        """
        Fit the rolling Ornstein-Uhlenbeck model.

        Performs rolling OU parameter estimation on spread (no intercept).
        """
        # Input validation
        y0 = np.asarray(self.y0, dtype=np.float64).squeeze()
        y1 = np.asarray(self.y1, dtype=np.float64).squeeze()
        beta = np.asarray(self.beta, dtype=np.float64)

        if y0.ndim != 1 or y1.ndim != 1:
            raise ValueError("y0 and y1 must be 1-dimensional")
        if len(y0) != len(y1):
            raise ValueError("y0 and y1 must have the same length")

        nobs = len(y0)

        # Initialize output arrays
        mu = np.full(nobs, np.nan)
        theta = np.full(nobs, np.nan)
        sigma = np.full(nobs, np.nan)
        half_life = np.full(nobs, np.nan)

        # Determine starting index
        if self.expanding:
            first_idx = max(
                self.min_nobs if self.min_nobs is not None else 2, self.lag + 5
            )
        else:
            first_idx = self.window

        # =========================================================================
        # STAGE 3: Rolling OU parameter estimation on spread
        # =========================================================================
        for t in range(first_idx - 1, nobs):
            # Get window bounds
            if self.expanding:
                w_start = 0
            else:
                w_start = t - self.window + 1
            w_end = t + 1

            # Compute spread for this window using pre-computed beta (no alpha)
            y0_window = y0[w_start:w_end]
            y1_window = y1[w_start:w_end]
            spread_window = y0_window - beta[t] * y1_window

            # Check for degenerate cases
            if len(spread_window) < 3 or np.std(spread_window, ddof=1) < 1e-10:
                continue

            # -----------------------------------------------------------------
            # OU Parameter Estimation on spread
            # Regression: spread_{t+1} = intercept + coef * spread_t + noise
            # -----------------------------------------------------------------
            try:
                # Set up OU regression: spread_{t+1} on spread_t
                spread_next = spread_window[1:]
                spread_lag = spread_window[:-1]

                # Add constant: [spread_lag, const]
                X_ou = np.column_stack([spread_lag, np.ones(len(spread_lag))])

                # OLS regression
                XtX_ou = X_ou.T @ X_ou
                Xty_ou = X_ou.T @ spread_next
                XtX_inv_ou = np.linalg.inv(XtX_ou)
                ou_params = XtX_inv_ou @ Xty_ou

                coef = ou_params[0]  # AR(1) coefficient
                intercept = ou_params[1]  # Intercept

                # For valid mean-reverting OU: coef must be in (0, 1)
                # coef = exp(-mu * DELTA_T)
                if 0 < coef < 1:
                    # Extract OU parameters
                    mu_val = -np.log(coef) / DELTA_T
                    theta_val = intercept / (1 - coef)

                    # Compute sigma from residuals
                    ou_resid = spread_next - X_ou @ ou_params
                    n_resid = len(ou_resid)
                    residual_var = (
                        np.sum(ou_resid**2) / (n_resid - 2) if n_resid > 2 else np.nan
                    )

                    if residual_var > 0 and not np.isnan(residual_var):
                        ou_residual_std = np.sqrt(residual_var)
                        # ou_residual_std^2 = sigma^2 * (1 - exp(-2*mu*dt)) / (2*mu)
                        factor = (1 - np.exp(-2 * mu_val * DELTA_T)) / (2 * mu_val)
                        if factor > 0:
                            sigma_val = ou_residual_std / np.sqrt(factor)

                            mu[t] = mu_val
                            theta[t] = theta_val
                            sigma[t] = sigma_val
                            half_life[t] = np.log(2) / mu_val

            except (np.linalg.LinAlgError, Exception):
                pass

        return RollingOrnsteinUhlenbeckResults(
            mu=mu,
            theta=theta,
            sigma=sigma,
            half_life=half_life,
        )

In [3]:
include_provider_asset_group_ids = [57, 4108, 3250]
# include_provider_asset_group_ids = [4108]
max_groups = 50
window_days = 7
window = window_days * 24 * 60
test_days = 120
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

Running pairs trading over period 2025-10-10 to 2026-02-14


In [4]:
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).where(
            models.ProviderAssetGroup.is_active.is_(True)
        )
    ).all()
    provider_asset_group_ids = random.sample(
        provider_asset_group_ids,
        max_groups - len(include_provider_asset_group_ids)
        if max_groups > len(include_provider_asset_group_ids)
        else 0,
    )
    provider_asset_group_ids = (
        include_provider_asset_group_ids + provider_asset_group_ids
    )
    provider_asset_group_ids = list(set(provider_asset_group_ids))
    provider_asset_group_ids = sorted(provider_asset_group_ids)
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

Provider asset group ids (count: 50): [29, 57, 191, 320, 379, 429, 494, 529, 536, 569, 592, 1010, 1104, 1144, 1156, 1285, 1547, 1618, 1806, 2101, 2138, 2193, 2338, 2378, 2393, 2403, 2437, 2479, 2695, 2724, 2763, 2997, 3211, 3250, 3251, 3471, 3601, 3681, 3743, 3832, 3847, 3867, 3871, 3963, 4017, 4031, 4073, 4108, 4119, 4171]


In [5]:
cluster = None
if not cluster:
    if CLUSTER_TYPE == "local":
        try:
            cluster.close()
        except:
            pass
        cluster = LocalCluster(
            name="local-cluster", n_workers=N_WORKERS, memory_limit="4GB"
        )
    elif CLUSTER_TYPE == "coiled":
        cluster = Cluster(
            name="coiled-cluster",
            n_workers=min(N_WORKERS * 10, 30),
            region="us-east-1",
            worker_memory="8GB",
            worker_cpu=2,
        )
        cluster.send_private_envs({"POSTGRES_URL": POSTGRES_URL})

In [6]:
client = cluster.get_client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 6
Total threads: 18,Total memory: 22.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61150,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61166,Total threads: 3
Dashboard: http://127.0.0.1:61168/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:61153,


In [7]:
db_semaphore = Semaphore(max_leases=N_CONCCURENT_DATABASE_CALLS, name="db_access")

In [8]:
@delayed
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_id: int,
    db_semaphore: Semaphore,
) -> pd.DataFrame:
    with db_semaphore:
        # Get the postgres URL.
        postgres_url = os.getenv("POSTGRES_URL")
        engine = create_engine(postgres_url)

        # Generate timeframe using pd.date_range
        time_frame_df = pd.DataFrame(
            {"timestamp": pd.date_range(start, end, freq="1min")}
        )

        # Use passed-in members_chunk (already filtered)
        members_df = pd.read_sql(
            select(
                models.ProviderAssetGroupMember.provider_asset_group_id,
                models.ProviderAssetGroupMember.order,
                models.ProviderAssetGroupMember.provider_id,
                models.ProviderAssetGroupMember.from_asset_id,
                models.ProviderAssetGroupMember.to_asset_id,
            ).where(
                models.ProviderAssetGroupMember.provider_asset_group_id
                == provider_asset_group_id
            ),
            engine,
        )

        # Cross join
        full_frame_df = time_frame_df.merge(members_df, how="cross")
        full_frame_df = full_frame_df.sort_values("timestamp")

        # Get market data.
        market_df: pd.DataFrame = pd.read_sql(
            select(
                models.ProviderAssetMarket.timestamp,
                models.ProviderAssetMarket.provider_id,
                models.ProviderAssetMarket.from_asset_id,
                models.ProviderAssetMarket.to_asset_id,
                models.ProviderAssetMarket.close,
            )
            .where(
                models.ProviderAssetMarket.timestamp.between(start, end),
                models.ProviderAssetMarket.from_asset_id.in_(
                    members_df["from_asset_id"].unique().tolist()
                ),
                models.ProviderAssetMarket.to_asset_id.in_(
                    members_df["to_asset_id"].unique().tolist()
                ),
            )
            .order_by(models.ProviderAssetMarket.timestamp),
            engine,
            parse_dates=["timestamp"],
        )
        market_df = market_df.astype(
            {
                "provider_id": "int64",
                "from_asset_id": "int64",
                "to_asset_id": "int64",
                "close": "float64",
            }
        )

        # Merge_asof
        full_market_frame = pd.merge_asof(
            full_frame_df,
            market_df,
            on="timestamp",
            by=["provider_id", "from_asset_id", "to_asset_id"],
            direction="backward",
        )

        # Split by order and create pairs - only keep essential columns
        close_1 = full_market_frame[full_market_frame["order"] == 1][
            ["timestamp", "provider_asset_group_id", "close"]
        ].rename(columns={"close": "close_1"})
        close_2 = full_market_frame[full_market_frame["order"] == 2][
            ["timestamp", "provider_asset_group_id", "close"]
        ].rename(columns={"close": "close_2"})

        # Merge to create pairs - only timestamp, close_1, close_2
        pairs = pd.merge(
            close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
        )

        # Keep only essential columns
        pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

        # Set index to provider_asset_group_id
        pairs = pairs.set_index("provider_asset_group_id")

        return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    db_semaphore: Semaphore,
    window_timedelta: dt.timedelta = dt.timedelta(days=7),
    partition_timedelta: dt.timedelta = dt.timedelta(days=1),
) -> dd.DataFrame:
    # Split time range into 1-day chunks
    provider_asset_group_ids = sorted(provider_asset_group_ids)

    # Based on the number of provider asset group ids, start and end times, calculate the number of partitions
    num_partitions = len(provider_asset_group_ids) * (
        (end - start) // partition_timedelta
    )
    print(f"Number of partitions: {num_partitions}")

    # Create delayed tasks: one per (provider_asset_group_id, day) combination
    # Each partition includes lookback data for rolling window calculations
    delayed_dfs = []
    for provider_asset_group_id in provider_asset_group_ids:
        for start_i in pd.date_range(start, end, freq=partition_timedelta):
            # Calculate the end of the chunk
            chunk_end = start_i + partition_timedelta
            if chunk_end > end:
                chunk_end = end

            # Calculate lookback start to include window_days of historical data
            chunk_start = start_i - window_timedelta

            # Load data with lookback (chunk_start to day_end) for rolling calculations
            # The partition will include lookback data, but we'll filter results to target day later
            delayed_dfs.append(
                load_pairs_trading_frame_chunk(
                    chunk_start, chunk_end, provider_asset_group_id, db_semaphore
                )
            )

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)
    pairs_trading_frame = pairs_trading_frame.reset_index()

    return pairs_trading_frame

In [9]:
pairs_trading_frame = get_pairs_trading_frame(
    start_time,
    end_time,
    provider_asset_group_ids,
    db_semaphore,
    dt.timedelta(days=7),
    dt.timedelta(days=14),
)

Number of partitions: 450


In [10]:
def rolling_ornstein_uhlenbeck(df: pd.DataFrame, window: dt.timedelta) -> pd.DataFrame:
    """Apply rolling Ornstein-Uhlenbeck to a DataFrame and return merged result with provider_asset_group_id as index."""
    # Copy the input DataFrame.
    output_df = df.copy()

    # Compute rolling Ornstein-Uhlenbeck
    cointegration_result = RollingCointegration(
        y0=output_df["close_1"].to_numpy(),
        y1=output_df["close_2"].to_numpy(),
        window=int(window.total_seconds() // 60),
    ).fit()

    # Create DataFrame with cointegration results indexed by timestamp
    timestamp_values = (
        output_df["timestamp"].values
        if hasattr(output_df["timestamp"], "values")
        else output_df["timestamp"]
    )
    cointegration_df = pd.DataFrame(
        {
            "beta": cointegration_result.beta,
            "pvalue": cointegration_result.pvalue,
            "residual_mean": cointegration_result.residual_mean,
            "residual_std": cointegration_result.residual_std,
        },
        index=timestamp_values,
    )
    cointegration_df.dropna(inplace=True)

    # Merge with original DataFrame
    output_df = output_df.merge(
        cointegration_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Compute the Ornstein-Uhlenbeck parameters
    ou_result = RollingOrnsteinUhlenbeck(
        beta=cointegration_result.beta,
        y0=df["close_1"].to_numpy(),
        y1=df["close_2"].to_numpy(),
        window=int(window.total_seconds() // 60),
    ).fit()

    # Create DataFrame with OU results indexed by timestamp
    ou_df = pd.DataFrame(
        {
            "mu": ou_result.mu,
            "sigma": ou_result.sigma,
            "theta": ou_result.theta,
            "half_life": ou_result.half_life,
        },
        index=timestamp_values,
    )
    ou_df.dropna(inplace=True)

    # Merge with original DataFrame
    output_df = output_df.merge(
        ou_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Reset index to drop timestamp index, then set provider_asset_group_id as index
    output_df = output_df.reset_index(drop=True)

    return output_df.set_index("provider_asset_group_id")


pairs_trading_frame = dd.map_partitions(
    rolling_ornstein_uhlenbeck,
    pairs_trading_frame,
    dt.timedelta(days=7),
    meta=pd.DataFrame(
        data={
            "timestamp": pd.Series([], dtype="datetime64[ns]"),
            "close_1": pd.Series([], dtype=float),
            "close_2": pd.Series([], dtype=float),
            "beta": pd.Series([], dtype=float),
            "pvalue": pd.Series([], dtype=float),
            "residual_mean": pd.Series([], dtype=float),
            "residual_std": pd.Series([], dtype=float),
            "mu": pd.Series([], dtype=float),
            "sigma": pd.Series([], dtype=float),
            "theta": pd.Series([], dtype=float),
            "half_life": pd.Series([], dtype=float),
        },
        index=pd.Index([], name="provider_asset_group_id", dtype="int64"),
    ),
)

TokenizationError: Object <function rolling_ornstein_uhlenbeck at 0x17520ce00> cannot be deterministically hashed. This likely indicates that the object cannot be serialized deterministically.

In [ ]:
def exit_level_partition(
    partition: pd.DataFrame,
    discount_rate=0.01,  # Or use your DISCOUNT_RATE
    transaction_cost=0.01,  # Or use your TRANSACTION_COST
    max_iter=50,
    tol=1e-7,
    n_grid=1000,
):
    """
    Compute exit_level for a partition of a Dask DataFrame. Only where pvalue is less than the threshold.
    """
    partition = partition.copy()

    # Only compute for rows where pvalue < PVALUE_THRESHOLD
    valid = partition["pvalue"] < PVALUE_THRESHOLD

    # Prepare output column initialized as NaN (for safety, for pandas not Dask)
    exit_level = np.full(len(partition), np.nan, dtype=float)

    # Only compute exit level for pvalue under the threshold.
    if valid.any():
        mu = partition.loc[valid, "mu"].to_numpy()
        sigma = partition.loc[valid, "sigma"].to_numpy()
        theta = partition.loc[valid, "theta"].to_numpy()
        idx = valid.values  # boolean mask (np.ndarray)
        exit_level[idx] = OrnsteinUhlenbeck.get_optimal_exit_level(
            mu=mu,
            sigma=sigma,
            theta=theta,
            discount_rate=discount_rate,
            transaction_cost=transaction_cost,
            max_iter=max_iter,
            tol=tol,
            n_grid=n_grid,
        )
    partition["exit_level"] = exit_level

    return partition


# Add the exit_level column via map_partitions
pairs_trading_frame = pairs_trading_frame.map_partitions(
    exit_level_partition,
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_iter=1000,
    n_grid=100,
    tol=1e-6,
)

In [ ]:
def entry_level_partition(
    partition: pd.DataFrame,
    discount_rate: float,
    transaction_cost: float,
    max_iter=1000,
    tol=1e-6,
    n_grid=100,
):
    """
    Dask map_partitions-compatible function to compute entry_level per partition,
    only for rows where mu, sigma, theta, and exit_level are not null
    (actually those with pvalue < PVALUE_THRESHOLD).
    """
    partition = partition.copy()

    # Only compute for rows where pvalue < PVALUE_THRESHOLD
    valid = partition["pvalue"] < PVALUE_THRESHOLD

    # Prepare output column initialized as NaN (for safety, for pandas not Dask)
    entry_level = np.full(len(partition), np.nan, dtype=float)

    # Only compute entry level for pvalue under the threshold.
    if valid.any():
        mu = partition.loc[valid, "mu"].to_numpy()
        sigma = partition.loc[valid, "sigma"].to_numpy()
        theta = partition.loc[valid, "theta"].to_numpy()
        exit_level = partition.loc[valid, "exit_level"].to_numpy()
        idx = valid.values  # boolean mask (np.ndarray)
        entry_level[idx] = OrnsteinUhlenbeck.get_optimal_entry_level(
            mu=mu,
            sigma=sigma,
            theta=theta,
            exit_level=exit_level,
            discount_rate=discount_rate,
            transaction_cost=transaction_cost,
            max_iter=max_iter,
            tol=tol,
            n_grid=n_grid,
        )
    partition["entry_level"] = entry_level

    return partition


# Add the entry_level column via map_partitions, respecting null logic
pairs_trading_frame: dd.DataFrame = pairs_trading_frame.map_partitions(
    entry_level_partition,
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_iter=1000,
    n_grid=100,
    tol=1e-6,
)

In [ ]:
@jit(nopython=True)
def compute_trades(
    timestamp: np.ndarray,
    close_1: np.ndarray,
    close_2: np.ndarray,
    beta: np.ndarray,
    spread: np.ndarray,
    pvalue: np.ndarray,
    entry_level: np.ndarray,
    exit_level: np.ndarray,
    loss_level: np.ndarray,
    threshold_pvalue: float,
):
    # Setup the output data structures.
    result = np.zeros_like(timestamp, dtype="int64")
    exit_reasons = np.zeros_like(
        timestamp, dtype="int64"
    )  # 0=none, 1=profit_target, 2=stop_loss
    n = result.shape[0]

    # Setup cache.
    trade_open = False
    trade_beta: float = None
    trade_exit_level: float = None
    trade_loss_level: float = None

    for i in range(n):
        if trade_open:
            spread_i = close_1[i] - trade_beta * close_2[i]
            if spread_i > trade_exit_level:
                result[i] = -1
                exit_reasons[i] = 1  # Profit target
                trade_open = False
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
            elif spread_i < trade_loss_level:
                result[i] = -1
                exit_reasons[i] = 2  # Stop loss
                trade_open = False
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
        else:
            if (
                not np.isnan(entry_level[i])
                and not np.isnan(exit_level[i])
                and (spread[i] < entry_level[i])
                and (pvalue[i] < threshold_pvalue)
                and (spread[i] > loss_level[i])
            ):
                result[i] = 1
                trade_open = True
                trade_beta = beta[i]
                trade_exit_level = exit_level[i]
                trade_loss_level = loss_level[i]

    return result, exit_reasons


def calculate_pnl_with_costs(
    position_size: Union[float, int, NDArray[np.float64]],
    entry_price: Union[float, NDArray[np.float64]],
    exit_price: Union[float, NDArray[np.float64]],
    entry_time: Union[pd.Timestamp, NDArray],
    exit_time: Union[pd.Timestamp, NDArray],
    commission_rate: float = 0.001,
    borrow_rate_annual: float = 0.05,
    is_long: Union[bool, NDArray[np.bool_]] = None,
) -> Union[float, NDArray[np.float64]]:
    """
    Calculate PnL with transaction costs for long and short positions.

    Parameters
    ----------
    position_size : scalar or array
        Number of shares (can be positive, negative, or use is_long flag)
        If negative, treated as short position (unless is_long overrides)
    entry_price : scalar or array
        Entry price in $/share
    exit_price : scalar or array
        Exit price in $/share
    entry_time : pd.Timestamp or array of timestamps
        Entry timestamp for each trade
    exit_time : pd.Timestamp or array of timestamps
        Exit timestamp for each trade
    commission_rate : float, default 0.001
        Commission rate per side (e.g., 0.001 = 0.1%)
    borrow_rate_annual : float, default 0.05
        Annualized borrow cost for short positions (e.g., 0.05 = 5%)
    is_long : bool or array, optional
        Explicitly specify if position is long (True) or short (False)
        If None, inferred from sign of position_size

    Returns
    -------
    pnl : scalar or array
        Net PnL in $ (same shape as inputs)
    """
    # Convert to arrays
    position_size = np.asarray(position_size, dtype=np.float64)
    entry_price = np.asarray(entry_price, dtype=np.float64)
    exit_price = np.asarray(exit_price, dtype=np.float64)

    # Convert timestamps to numpy datetime64 if needed
    if isinstance(entry_time, pd.Timestamp):
        entry_time = np.array([entry_time], dtype="datetime64[ns]")
    elif isinstance(entry_time, (list, pd.DatetimeIndex)):
        entry_time = pd.to_datetime(entry_time).values
    else:
        entry_time = np.asarray(entry_time, dtype="datetime64[ns]")

    if isinstance(exit_time, pd.Timestamp):
        exit_time = np.array([exit_time], dtype="datetime64[ns]")
    elif isinstance(exit_time, (list, pd.DatetimeIndex)):
        exit_time = pd.to_datetime(exit_time).values
    else:
        exit_time = np.asarray(exit_time, dtype="datetime64[ns]")

    # Calculate holding period in days (including fractional days)
    time_delta = exit_time - entry_time
    holding_days = time_delta / np.timedelta64(1, "D")
    holding_days = holding_days.astype(np.float64)

    # Determine if positions are long or short
    if is_long is None:
        # Infer from sign of position_size
        is_long_arr = position_size >= 0
        abs_pos = np.abs(position_size)
    else:
        is_long_arr = np.asarray(is_long, dtype=bool)
        abs_pos = np.abs(position_size)

    # Calculate gross PnL
    pnl = np.where(
        is_long_arr,
        abs_pos * (exit_price - entry_price),  # Long PnL
        abs_pos * (entry_price - exit_price),  # Short PnL
    )

    # Commissions (both entry and exit)
    commissions = abs_pos * entry_price * commission_rate
    commissions += abs_pos * exit_price * commission_rate

    # Borrow cost (only for shorts)
    daily_rate = borrow_rate_annual / 365.0
    borrow_cost = np.where(
        ~is_long_arr,  # Only for shorts
        abs_pos * entry_price * daily_rate * holding_days,
        0.0,
    )

    net_pnl = pnl - commissions - borrow_cost

    # Return scalar if inputs were scalar
    return float(net_pnl) if net_pnl.ndim == 0 else net_pnl


def compute_pnl(
    df: pd.DataFrame,
    threshold_pvalue: float,
    cash_allocation: float,
    loss_percentage: float = 0.02,
):
    """
    Compute PnL using the beta from cointegration for proper hedging.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with trade signals and prices
    threshold_pvalue : float
        P-value threshold for entering trades
    cash_allocation : float
        Total dollar amount to allocate per trade (split between long and short legs)
    loss_percentage : float, default 0.02
        Maximum acceptable loss as percentage of TOTAL capital allocation
    """
    df = df.copy()

    # Calculate position sizes based on total capital allocation
    # We want: long_shares * close_1 + beta * long_shares * close_2 = cash_allocation
    # Therefore: long_shares = cash_allocation / (close_1 + beta * close_2)
    long_shares = cash_allocation / (df["close_1"] + df["beta"] * df["close_2"])

    # Total exposure is now simply cash_allocation (constant across all pairs)
    total_exposure = cash_allocation

    # Calculate max acceptable dollar loss based on TOTAL exposure
    max_dollar_loss = loss_percentage * total_exposure

    # How much can the spread move against us before we hit this loss?
    # When spread moves by $1 against us, we lose long_shares dollars
    adverse_spread_change = max_dollar_loss / long_shares

    # Loss level is BELOW entry (spread falling further = bad)
    df["loss_level"] = df["entry_level"] - adverse_spread_change

    # Compute the enter and exit trades with dynamic loss levels
    trades, exit_reasons = compute_trades(
        df["timestamp"].to_numpy(),
        df["close_1"].to_numpy(),
        df["close_2"].to_numpy(),
        df["beta"].to_numpy(),
        df["spread"].to_numpy(),
        df["pvalue"].to_numpy(),
        df["entry_level"].to_numpy(),
        df["exit_level"].to_numpy(),
        df["loss_level"].to_numpy(),
        threshold_pvalue,
    )

    # Only take data from the frame where we are either entering or exiting a trade.
    actual_trades = df[trades != 0].copy()
    actual_exit_reasons = exit_reasons[trades != 0]

    # If we have an odd number of trades, the last position is still open
    if len(actual_trades) % 2 != 0:
        last_row = df.iloc[-1].copy()
        last_row_df = pd.DataFrame([last_row])
        actual_trades = pd.concat([actual_trades, last_row_df], ignore_index=True)
        # Mark the forced exit as reason 3 (end of data)
        actual_exit_reasons = np.append(actual_exit_reasons, 3)

    # Re-shape the arrays
    ou_mu = actual_trades["mu"].to_numpy().reshape(-1, 2)
    ou_sigma = actual_trades["sigma"].to_numpy().reshape(-1, 2)
    ou_theta = actual_trades["theta"].to_numpy().reshape(-1, 2)
    close_1_prices = actual_trades["close_1"].to_numpy().reshape(-1, 2)
    close_2_prices = actual_trades["close_2"].to_numpy().reshape(-1, 2)
    times = actual_trades["timestamp"].to_numpy().reshape(-1, 2)

    # Reshape exit reasons (only exits have reasons, entries are 0)
    exit_reasons_reshaped = actual_exit_reasons.reshape(-1, 2)
    exit_reasons_codes = exit_reasons_reshaped[:, 1]  # Take the exit (second element)

    # Map exit reason codes to strings
    exit_reason_map = {0: "None", 1: "Profit Target", 2: "Stop Loss", 3: "End of Data"}
    exit_reasons_str = np.array([exit_reason_map[code] for code in exit_reasons_codes])

    # Get the beta values at entry points
    betas = actual_trades["beta"].to_numpy().reshape(-1, 2)
    entry_level = actual_trades["entry_level"].to_numpy().reshape(-1, 2)
    exit_level = actual_trades["exit_level"].to_numpy().reshape(-1, 2)
    entry_betas = betas[:, 0]
    entry_entry_levels = entry_level[:, 0]
    entry_exit_levels = exit_level[:, 0]

    # Get the spreads.
    spread_entry = close_1_prices[:, 0] - entry_betas * close_2_prices[:, 0]
    spread_exit = close_1_prices[:, 1] - entry_betas * close_2_prices[:, 1]

    # Position sizing - split total capital between long and short legs
    long_position_sizes = cash_allocation / (
        close_1_prices[:, 0] + entry_betas * close_2_prices[:, 0]
    )
    short_position_sizes = entry_betas * long_position_sizes

    # Compute PnL for each leg
    long_pnl = calculate_pnl_with_costs(
        long_position_sizes,
        close_1_prices[:, 0],
        close_1_prices[:, 1],
        times[:, 0],
        times[:, 1],
        is_long=True,
    )

    short_pnl = calculate_pnl_with_costs(
        short_position_sizes,
        close_2_prices[:, 0],
        close_2_prices[:, 1],
        times[:, 0],
        times[:, 1],
        borrow_rate_annual=0.05,
        is_long=False,
    )

    return pd.DataFrame(
        {
            "entry_time": pd.Series(times[:, 0], dtype="datetime64[ns]"),
            "exit_time": pd.Series(times[:, 1], dtype="datetime64[ns]"),
            "exit_reason": pd.Series(exit_reasons_str, dtype="str"),
            "ou_mu_entry": pd.Series(ou_mu[:, 0], dtype="float64"),
            "ou_mu_exit": pd.Series(ou_mu[:, 1], dtype="float64"),
            "ou_sigma_entry": pd.Series(ou_sigma[:, 0], dtype="float64"),
            "ou_sigma_exit": pd.Series(ou_sigma[:, 1], dtype="float64"),
            "ou_theta_entry": pd.Series(ou_theta[:, 0], dtype="float64"),
            "ou_theta_exit": pd.Series(ou_theta[:, 1], dtype="float64"),
            "short_entry_price": pd.Series(close_2_prices[:, 0], dtype="float64"),
            "short_exit_price": pd.Series(close_2_prices[:, 1], dtype="float64"),
            "short_pnl": pd.Series(short_pnl, dtype="float64"),
            "short_position_size": pd.Series(short_position_sizes, dtype="float64"),
            "long_entry_price": pd.Series(close_1_prices[:, 0], dtype="float64"),
            "long_exit_price": pd.Series(close_1_prices[:, 1], dtype="float64"),
            "long_pnl": pd.Series(long_pnl, dtype="float64"),
            "long_position_size": pd.Series(long_position_sizes, dtype="float64"),
            "hedge_ratio": pd.Series(entry_betas, dtype="float64"),
            "spread_entry": pd.Series(spread_entry, dtype="float64"),
            "spread_exit": pd.Series(spread_exit, dtype="float64"),
            "entry_level": pd.Series(entry_entry_levels, dtype="float64"),
            "exit_level": pd.Series(entry_exit_levels, dtype="float64"),
        }
    )

In [ ]:
pairs_trading_frame = pairs_trading_frame.loc[pairs_trading_frame["beta"].notnull()]
pairs_trading_frame["spread"] = (
    pairs_trading_frame["close_1"]
    - pairs_trading_frame["beta"] * pairs_trading_frame["close_2"]
)

In [ ]:
# Setup the meta for the output.
meta = pd.DataFrame(
    {
        "entry_time": pd.Series([], dtype="datetime64[ns]"),
        "exit_time": pd.Series([], dtype="datetime64[ns]"),
        "exit_reason": pd.Series([], dtype="str"),
        "ou_mu_entry": pd.Series([], dtype="float64"),
        "ou_mu_exit": pd.Series([], dtype="float64"),
        "ou_sigma_entry": pd.Series([], dtype="float64"),
        "ou_sigma_exit": pd.Series([], dtype="float64"),
        "ou_theta_entry": pd.Series([], dtype="float64"),
        "ou_theta_exit": pd.Series([], dtype="float64"),
        "short_entry_price": pd.Series([], dtype="float64"),
        "short_exit_price": pd.Series([], dtype="float64"),
        "short_pnl": pd.Series([], dtype="float64"),
        "short_position_size": pd.Series([], dtype="float64"),
        "long_entry_price": pd.Series([], dtype="float64"),
        "long_exit_price": pd.Series([], dtype="float64"),
        "long_pnl": pd.Series([], dtype="float64"),
        "long_position_size": pd.Series([], dtype="float64"),
        "hedge_ratio": pd.Series([], dtype="float64"),
        "spread_entry": pd.Series([], dtype="float64"),
        "spread_exit": pd.Series([], dtype="float64"),
        "entry_level": pd.Series([], dtype="float64"),
        "exit_level": pd.Series([], dtype="float64"),
    }
).set_index(pd.Index([], name="provider_asset_group_id"))

# Compute the frame with current parameters
print(f"\nComputing results with PERCENT_LOSS={PERCENT_LOSS}...")
results_df = (
    pairs_trading_frame.groupby("provider_asset_group_id")[
        [
            "timestamp",
            "close_1",
            "close_2",
            "beta",
            "spread",
            "pvalue",
            "mu",
            "sigma",
            "theta",
            "entry_level",
            "exit_level",
        ]
    ]
    .apply(
        lambda df: compute_pnl(df, PVALUE_THRESHOLD, CASH_ALLOCATION, PERCENT_LOSS),
        meta=meta,
    )
    .compute()
)

print(f"✓ Results computed! Found {len(results_df)} trades.")

In [ ]:
# Helper function to calculate statistics for any subset of trades
def calculate_statistics(all_trades_subset, dataset_name="ALL TRADES"):
    """Calculate comprehensive statistics for a given subset of trades"""

    # Reset index to turn provider_asset_group_id into a column
    results_display = all_trades_subset.reset_index(
        level="provider_asset_group_id"
        if "provider_asset_group_id" in all_trades_subset.index.names
        else None,
        drop=False,
    )

    # Calculate dollar exposures BEFORE aggregation
    results_display["long_dollar_exposure"] = (
        results_display["long_position_size"] * results_display["long_entry_price"]
    )
    results_display["short_dollar_exposure"] = (
        results_display["short_position_size"] * results_display["short_entry_price"]
    )
    results_display["total_dollar_exposure"] = (
        results_display["long_dollar_exposure"]
        + results_display["short_dollar_exposure"]
    )

    # Calculate total PnL
    results_display["total_pnl"] = (
        results_display["long_pnl"] + results_display["short_pnl"]
    )

    # Calculate percentage returns based on trade-specific exposure
    results_display["long_pct"] = (
        results_display["long_pnl"] / results_display["long_dollar_exposure"]
    ) * 100
    results_display["short_pct"] = (
        results_display["short_pnl"] / results_display["short_dollar_exposure"]
    ) * 100
    results_display["total_pct"] = (
        results_display["total_pnl"] / results_display["total_dollar_exposure"]
    ) * 100

    # Calculate constant capital allocation per pair (average exposure per trade)
    pair_capital = results_display.groupby("provider_asset_group_id")[
        "total_dollar_exposure"
    ].mean()

    # Add constant capital to each trade
    results_display = results_display.merge(
        pair_capital.rename("pair_constant_capital"),
        left_on="provider_asset_group_id",
        right_index=True,
        how="left",
    )

    # Calculate percentage gain based on constant capital allocation
    results_display["pct_on_capital"] = (
        results_display["total_pnl"] / results_display["pair_constant_capital"]
    ) * 100

    # Store all trades before aggregation
    all_trades_detail = results_display.copy()

    # If there are duplicate provider_asset_group_ids, aggregate them
    if (
        "provider_asset_group_id" in results_display.columns
        and results_display["provider_asset_group_id"].duplicated().any()
    ):
        # Calculate win/loss counts per group BEFORE aggregation
        win_loss_counts = results_display.groupby("provider_asset_group_id").agg(
            num_trades=("total_pnl", "size"),
            num_wins=("total_pnl", lambda x: (x > 0).sum()),
            num_losses=("total_pnl", lambda x: (x < 0).sum()),
        )

        # Aggregate by provider_asset_group_id
        results_display = results_display.groupby(
            "provider_asset_group_id", as_index=False
        ).agg(
            {
                "long_pnl": "sum",
                "short_pnl": "sum",
                "total_pnl": "sum",
                "long_dollar_exposure": "sum",
                "short_dollar_exposure": "sum",
                "total_dollar_exposure": "sum",
                "hedge_ratio": "mean",
                "pair_constant_capital": "first",  # Same for all trades in a pair
                "pct_on_capital": "sum",  # Sum up percentage gains
            }
        )

        # Merge win/loss counts back
        results_display = results_display.merge(
            win_loss_counts, on="provider_asset_group_id", how="left"
        )

        # Recalculate percentages based on total exposure after aggregation
        results_display["long_pct"] = (
            results_display["long_pnl"] / results_display["long_dollar_exposure"]
        ) * 100
        results_display["short_pct"] = (
            results_display["short_pnl"] / results_display["short_dollar_exposure"]
        ) * 100
        results_display["total_pct"] = (
            results_display["total_pnl"] / results_display["total_dollar_exposure"]
        ) * 100
    else:
        # Add win/loss counts for non-aggregated data
        results_display["num_trades"] = 1
        results_display["num_wins"] = (results_display["total_pnl"] > 0).astype(int)
        results_display["num_losses"] = (results_display["total_pnl"] < 0).astype(int)

    # Calculate summary statistics
    winning_trades = results_display["total_pnl"] > 0
    losing_trades = results_display["total_pnl"] < 0

    avg_win = (
        results_display.loc[winning_trades, "total_pnl"].mean()
        if winning_trades.sum() > 0
        else 0
    )
    avg_loss = (
        results_display.loc[losing_trades, "total_pnl"].mean()
        if losing_trades.sum() > 0
        else 0
    )

    total_wins = (
        results_display.loc[winning_trades, "total_pnl"].sum()
        if winning_trades.sum() > 0
        else 0
    )
    total_losses = (
        abs(results_display.loc[losing_trades, "total_pnl"].sum())
        if losing_trades.sum() > 0
        else 0
    )
    profit_factor = total_wins / total_losses if total_losses > 0 else float("inf")

    sharpe = (
        results_display["total_pnl"].mean() / results_display["total_pnl"].std()
        if results_display["total_pnl"].std() > 0
        else 0
    )

    # Market neutrality metrics
    avg_hedge_ratio = results_display["hedge_ratio"].mean()
    exposure_ratio = (
        results_display["short_dollar_exposure"].sum()
        / results_display["long_dollar_exposure"].sum()
    )
    net_exposure = (
        results_display["long_dollar_exposure"]
        - results_display["short_dollar_exposure"]
    )
    long_short_corr = results_display["long_pnl"].corr(results_display["short_pnl"])
    long_short_ratio = (
        results_display["long_pnl"].sum() / results_display["short_pnl"].sum()
        if results_display["short_pnl"].sum() != 0
        else float("inf")
    )

    # Total trade counts across all pairs
    total_num_trades = results_display["num_trades"].sum()
    total_num_wins = results_display["num_wins"].sum()
    total_num_losses = results_display["num_losses"].sum()

    # Calculate total constant capital across all pairs
    total_constant_capital = results_display["pair_constant_capital"].sum()

    # Calculate overall return on constant capital
    overall_pct_on_capital = (
        (results_display["total_pnl"].sum() / total_constant_capital) * 100
        if total_constant_capital > 0
        else 0
    )

    # Calculate annualized return
    backtest_start = all_trades_detail["entry_time"].min()
    backtest_end = all_trades_detail["exit_time"].max()
    backtest_days = (backtest_end - backtest_start).total_seconds() / (60 * 60 * 24)
    total_return = overall_pct_on_capital / 100  # Convert to decimal
    annualized_return = (
        ((1 + total_return) ** (365 / backtest_days) - 1) * 100
        if backtest_days > 0
        else 0
    )

    # Exit reason breakdown
    exit_reason_counts = (
        all_trades_detail["exit_reason"].value_counts()
        if "exit_reason" in all_trades_detail.columns
        else pd.Series()
    )

    # Create overall summary row
    overall_summary = pd.DataFrame(
        {
            "provider_asset_group_id": ["OVERALL"],
            "long_pnl": [results_display["long_pnl"].sum()],
            "short_pnl": [results_display["short_pnl"].sum()],
            "total_pnl": [results_display["total_pnl"].sum()],
            "long_dollar_exposure": [results_display["long_dollar_exposure"].sum()],
            "short_dollar_exposure": [results_display["short_dollar_exposure"].sum()],
            "total_dollar_exposure": [results_display["total_dollar_exposure"].sum()],
            "long_pct": [
                (
                    results_display["long_pnl"].sum()
                    / results_display["long_dollar_exposure"].sum()
                )
                * 100
            ],
            "short_pct": [
                (
                    results_display["short_pnl"].sum()
                    / results_display["short_dollar_exposure"].sum()
                )
                * 100
            ],
            "total_pct": [
                (
                    results_display["total_pnl"].sum()
                    / results_display["total_dollar_exposure"].sum()
                )
                * 100
            ],
            "hedge_ratio": [avg_hedge_ratio],
            "pair_constant_capital": [total_constant_capital],
            "pct_on_capital": [overall_pct_on_capital],
            "num_trades": [total_num_trades],
            "num_wins": [total_num_wins],
            "num_losses": [total_num_losses],
        }
    )

    # Combine aggregated pairs with overall summary
    results_with_summary = pd.concat(
        [results_display, overall_summary], ignore_index=True
    )

    # Create comprehensive summary table
    summary_data = {
        "Metric": [
            "Dataset",
            "Backtest Period (days)",
            "Total PnL",
            "Total Constant Capital",
            "Return on Capital",
            "Annualized Return",
            "Total Trades",
            "Number of Pairs",
            "Winning Pairs",
            "Losing Pairs",
            "Win Rate (Pairs)",
            "Total Winning Trades",
            "Total Losing Trades",
            "Win Rate (Trades)",
            "Average Win",
            "Average Loss",
            "Profit Factor",
            "Sharpe Ratio",
            "Average Hedge Ratio",
            "Exposure Ratio (S/L)",
            "Long/Short PnL Corr",
            "Long/Short PnL Ratio",
        ],
        "Value": [
            dataset_name,
            f"{backtest_days:.1f}",
            f"${results_display['total_pnl'].sum():,.2f}",
            f"${total_constant_capital:,.2f}",
            f"{overall_pct_on_capital:+.2f}%",
            f"{annualized_return:+.2f}%",
            f"{total_num_trades:,}",
            f"{len(results_display):,}",
            f"{winning_trades.sum():,}",
            f"{losing_trades.sum():,}",
            f"{winning_trades.sum()}/{len(results_display)} ({winning_trades.mean() * 100:.1f}%)",
            f"{total_num_wins:,}",
            f"{total_num_losses:,}",
            f"{total_num_wins}/{total_num_trades} ({total_num_wins / total_num_trades * 100:.1f}%)"
            if total_num_trades > 0
            else "N/A",
            f"${avg_win:,.2f}",
            f"${avg_loss:,.2f}",
            f"{profit_factor:.2f}",
            f"{sharpe:.2f}",
            f"{avg_hedge_ratio:.4f}",
            f"{exposure_ratio:.4f}",
            f"{long_short_corr:.3f}",
            f"{long_short_ratio:.3f}",
        ],
    }

    summary_df = pd.DataFrame(summary_data)

    return {
        "summary_df": summary_df,
        "results_display": results_display,
        "results_with_summary": results_with_summary,
        "all_trades_detail": all_trades_detail,
        "exit_reason_counts": exit_reason_counts,
    }


# Calculate statistics for ALL trades (with default PERCENT_LOSS)
print("=" * 120)
print("CALCULATING STATISTICS FOR ALL TRADES (INCLUDING 'END OF DATA' EXITS)")
print("=" * 120)
all_stats = calculate_statistics(results_df.copy(), "ALL TRADES")

# Calculate statistics EXCLUDING "End of Data" exits
print("\n" + "=" * 120)
print("CALCULATING STATISTICS EXCLUDING 'END OF DATA' EXITS")
print("=" * 120)
results_df_no_eod = results_df[results_df["exit_reason"] != "End of Data"].copy()
no_eod_stats = calculate_statistics(results_df_no_eod, "EXCLUDING 'END OF DATA'")

# Display ALL TRADES summary
print("\n" + "=" * 120)
print("📊 BACKTEST SUMMARY STATISTICS - ALL TRADES")
print("=" * 120 + "\n")

styled_summary_all = (
    all_stats["summary_df"]
    .style.set_properties(
        **{
            "text-align": "left",
            "font-weight": "bold",
            "color": "#F8F8F2",
            "background-color": "#272822",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("font-weight", "bold"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "tr:nth-child(even)",
                "props": [("background-color", "#3E3D32")],
            },
            {
                "selector": "tr:nth-child(odd)",
                "props": [("background-color", "#272822")],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("📊 BACKTEST SUMMARY STATISTICS - ALL TRADES")
)

display(styled_summary_all)

# Display exit reason breakdown for ALL trades
print("\n" + "=" * 120)
print("EXIT REASON BREAKDOWN - ALL TRADES")
print("=" * 120)
for reason, count in all_stats["exit_reason_counts"].items():
    pct = count / len(all_stats["all_trades_detail"]) * 100
    print(f"  {reason:20s}: {count:4d} trades ({pct:5.1f}%)")

# Display EXCLUDING "End of Data" summary
print("\n" + "=" * 120)
print("📊 BACKTEST SUMMARY STATISTICS - EXCLUDING 'END OF DATA' EXITS")
print("=" * 120 + "\n")

styled_summary_no_eod = (
    no_eod_stats["summary_df"]
    .style.set_properties(
        **{
            "text-align": "left",
            "font-weight": "bold",
            "color": "#F8F8F2",
            "background-color": "#272822",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("font-weight", "bold"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "tr:nth-child(even)",
                "props": [("background-color", "#3E3D32")],
            },
            {
                "selector": "tr:nth-child(odd)",
                "props": [("background-color", "#272822")],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("📊 BACKTEST SUMMARY STATISTICS - EXCLUDING 'END OF DATA' EXITS")
)

display(styled_summary_no_eod)

# Display exit reason breakdown for EXCLUDING "End of Data"
print("\n" + "=" * 120)
print("EXIT REASON BREAKDOWN - EXCLUDING 'END OF DATA' EXITS")
print("=" * 120)
for reason, count in no_eod_stats["exit_reason_counts"].items():
    pct = count / len(no_eod_stats["all_trades_detail"]) * 100
    print(f"  {reason:20s}: {count:4d} trades ({pct:5.1f}%)")

# Display aggregated results by pair (using ALL trades)
print("\n" + "=" * 120)
print("AGGREGATED RESULTS BY ASSET PAIR (with OVERALL summary) - ALL TRADES")
print("=" * 120 + "\n")

results_with_summary_display = all_stats["results_with_summary"].reset_index(drop=True)


# Function to apply color-coded text based on value
def apply_pnl_coloring(val, vmin, vmax):
    """Apply background gradient AND appropriate text color"""
    import matplotlib.colors as mcolors
    import matplotlib as mpl

    # Normalize value
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colormap = mpl.colormaps["RdYlGn"]
    rgba = colormap(norm(val))

    # Calculate luminance
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]

    # Use dark text for light backgrounds, light text for dark backgrounds
    text_color = "#000000" if luminance > 0.5 else "#F8F8F2"
    bg_color = mcolors.rgb2hex(rgba[:3])

    return f"background-color: {bg_color}; color: {text_color}"


def style_pnl_columns(df, col, vmin, vmax):
    """Style PnL columns with proper text contrast"""
    return df[col].apply(lambda v: apply_pnl_coloring(v, vmin, vmax))


# Dark mode styling for aggregated results with dynamic text color
styled_aggregated = (
    results_with_summary_display.style.format(
        {
            "long_pnl": "${:,.2f}",
            "short_pnl": "${:,.2f}",
            "total_pnl": "${:,.2f}",
            "long_pct": "{:+.2f}%",
            "short_pct": "{:+.2f}%",
            "total_pct": "{:+.2f}%",
            "pct_on_capital": "{:+.2f}%",
            "hedge_ratio": "{:.4f}",
            "long_dollar_exposure": "${:,.2f}",
            "short_dollar_exposure": "${:,.2f}",
            "total_dollar_exposure": "${:,.2f}",
            "pair_constant_capital": "${:,.2f}",
            "num_trades": "{:,.0f}",
            "num_wins": "{:,.0f}",
            "num_losses": "{:,.0f}",
        }
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "long_pnl", -1000, 1000
        ),
        subset=["long_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "short_pnl", -1000, 1000
        ),
        subset=["short_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "total_pnl", -1000, 1000
        ),
        subset=["total_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "long_pct", -10, 10),
        subset=["long_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "short_pct", -10, 10),
        subset=["short_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "total_pct", -10, 10),
        subset=["total_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "pct_on_capital", -20, 20
        ),
        subset=["pct_on_capital"],
    )
    .apply(
        lambda x: [
            "font-weight: bold; background-color: #49483E; color: #F8F8F2"
            if v == "OVERALL"
            else "color: #F8F8F2"
            for v in x
        ],
        subset=["provider_asset_group_id"],
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("Aggregated Performance by Asset Pair - ALL TRADES")
)

display(styled_aggregated)

print("\n" + "=" * 120)
print("ALL INDIVIDUAL TRADES")
print("=" * 120 + "\n")

# Add outcome indicator column for easier scanning
all_trades = all_stats["all_trades_detail"].copy()
all_trades["outcome"] = all_trades["total_pnl"].apply(
    lambda x: "✓ WIN" if x > 0 else ("✗ LOSS" if x < 0 else "- BREAK")
)

# Select and reorder columns for better readability
trades_display_cols = [
    "provider_asset_group_id",
    "entry_time",
    "exit_time",
    "exit_reason",
    "outcome",
    "total_pnl",
    "total_pct",
    "pct_on_capital",
    "long_entry_price",
    "long_exit_price",
    "long_position_size",
    "long_pnl",
    "long_pct",
    "short_entry_price",
    "short_exit_price",
    "short_position_size",
    "short_pnl",
    "short_pct",
    "hedge_ratio",
    "total_dollar_exposure",
    "pair_constant_capital",
]

# Check which columns exist and filter
available_cols = [col for col in trades_display_cols if col in all_trades.columns]
all_trades_display = all_trades[available_cols].reset_index(drop=True)


# Dark mode styling for all trades with dynamic text color
def highlight_outcome(row):
    """Color code the outcome column"""
    result = [""] * len(row)
    for idx, col in enumerate(row.index):
        if col == "outcome":
            if row[col] == "✓ WIN":
                result[idx] = (
                    "background-color: #2D5016; color: #A6E22E; font-weight: bold"
                )
            elif row[col] == "✗ LOSS":
                result[idx] = (
                    "background-color: #5C1F1F; color: #F92672; font-weight: bold"
                )
        elif col == "exit_reason":
            # Color code exit reasons
            if row[col] == "Profit Target":
                result[idx] = "background-color: #1E4D2B; color: #A6E22E"
            elif row[col] == "Stop Loss":
                result[idx] = "background-color: #4D1E1E; color: #F92672"
            elif row[col] == "End of Data":
                result[idx] = "background-color: #3E3D32; color: #FD971F"
    return result


def style_non_gradient_columns(row):
    """Apply default colors to non-gradient columns"""
    result = [""] * len(row)
    gradient_cols = [
        "long_pnl",
        "short_pnl",
        "total_pnl",
        "long_pct",
        "short_pct",
        "total_pct",
        "pct_on_capital",
        "outcome",
        "exit_reason",
    ]
    for idx, col in enumerate(row.index):
        if col not in gradient_cols:
            result[idx] = "color: #F8F8F2; white-space: nowrap"
    return result


styled_all_trades = (
    all_trades_display.style.format(
        {
            "entry_time": lambda x: x.strftime("%Y-%m-%d %H:%M") if pd.notna(x) else "",
            "exit_time": lambda x: x.strftime("%Y-%m-%d %H:%M") if pd.notna(x) else "",
            "long_entry_price": "${:,.2f}",
            "long_exit_price": "${:,.2f}",
            "long_position_size": "{:,.2f}",
            "short_entry_price": "${:,.2f}",
            "short_exit_price": "${:,.2f}",
            "short_position_size": "{:,.2f}",
            "long_pnl": "${:,.2f}",
            "short_pnl": "${:,.2f}",
            "total_pnl": "${:,.2f}",
            "long_pct": "{:+.2f}%",
            "short_pct": "{:+.2f}%",
            "total_pct": "{:+.2f}%",
            "pct_on_capital": "{:+.2f}%",
            "hedge_ratio": "{:.4f}",
            "total_dollar_exposure": "${:,.2f}",
            "pair_constant_capital": "${:,.2f}",
        }
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "long_pnl", -500, 500),
        subset=["long_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "short_pnl", -500, 500),
        subset=["short_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "total_pnl", -500, 500),
        subset=["total_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "long_pct", -5, 5),
        subset=["long_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "short_pct", -5, 5),
        subset=["short_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "total_pct", -5, 5),
        subset=["total_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "pct_on_capital", -10, 10),
        subset=["pct_on_capital"],
    )
    .apply(highlight_outcome, axis=1)
    .apply(style_non_gradient_columns, axis=1)
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                    ("white-space", "nowrap"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("All Individual Trades")
)

display(styled_all_trades)

print(f"\n{'=' * 120}")
print("\n💡 TIP: To quickly test different PERCENT_LOSS values, use:")
print(
    "   backtest_results = run_backtest_with_params(percent_loss=0.05)  # 5% stop loss"
)
print(
    "   backtest_results = run_backtest_with_params(percent_loss=0.15)  # 15% stop loss"
)
print("   This will reuse the persisted pairs_trading_frame and run much faster!")

In [ ]:
# Test with 5% stop loss
results_5pct = run_backtest_with_params(percent_loss=0.05)
display(results_5pct["all_stats"]["summary_df"])
display(results_5pct["no_eod_stats"]["summary_df"])

# Test with 15% stop loss
results_15pct = run_backtest_with_params(percent_loss=0.15)
display(results_15pct["all_stats"]["summary_df"])

# Compare multiple scenarios
for pct_loss in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    result = run_backtest_with_params(percent_loss=pct_loss)
    print(f"\nPERCENT_LOSS={pct_loss:.0%}")
    print(
        f"  Total PnL: ${result['all_stats']['results_display']['total_pnl'].sum():,.2f}"
    )
    print(
        f"  Win Rate: {result['all_stats']['results_display']['num_wins'].sum()}/{result['all_stats']['results_display']['num_trades'].sum()}"
    )